# 08 · Sequence Models: RNNs, LSTMs and GRUs

**Part IV — Sequences and language** · Estimated time: ~120 min

## Learning objectives

- Represent sequential data as `(batch, time, features)` tensors and classify sequence tasks by the shapes of their inputs and outputs.
- Implement a vanilla recurrent neural network from scratch and verify it numerically against `nn.RNN`.
- Derive backpropagation through time and explain, through the product of Jacobians, why gradients vanish or explode.
- Measure the decay of gradients over time numerically and apply truncated BPTT and gradient clipping.
- Implement an LSTM cell from scratch, check it against `nn.LSTMCell`, and explain why its additive cell state preserves gradients; compare it with the GRU.
- Train a character-level language model with truncated BPTT, evaluate it with perplexity and sample from it with temperature.
- Forecast a time series with an LSTM using correctly fitted scaling, classify variable-length sequences with packing, and build a sequence-to-sequence model trained with teacher forcing.

## Prerequisites

- Notebook 03 · Tensors and Autograd (autograd, `torch.autograd.grad`, `nn.Module`)
- Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization (Adam, gradient clipping, vanishing gradients in deep networks)
- Notebook 05 · The PyTorch Workflow and Best Practices (`Dataset`/`DataLoader`, training loops)

## Contents

1. Sequential data and why order matters
2. The vanilla RNN from scratch
3. Backpropagation through time
4. Gated architectures: LSTM and GRU
5. Experiment: learning long-range dependencies
6. A character-level language model on Tiny Shakespeare
7. Time-series forecasting
8. Sequence classification with variable-length inputs
9. Seq2seq and the bottleneck

Exercises · Summary · Further reading

In [ ]:
import math, os, random, time, re
import urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

# FAST_RUN = True shrinks every training budget so the whole notebook runs in a few
# minutes on a CPU. Set it to False for the full experiments (recommended on a GPU).
FAST_RUN = False
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

## 1. Sequential data and why order matters

Everything we have built so far consumed inputs of a *fixed* shape: a vector of features for the MLPs of Notebooks 02–05, an image of fixed height and width for the CNNs of Notebooks 06–07. A great deal of the world's data does not come that way. Text is a sequence of characters or words whose length varies from a tweet to a novel. Audio is a sequence of samples, a sensor log is a sequence of readings, a genome is a sequence of bases, a chess game is a sequence of moves. What these have in common is not only that the length varies, but that **order carries information**. "The dog bit the man" and "The man bit the dog" contain the same words; a heartbeat that speeds up is different from one that slows down even if the set of measured intervals is identical.

This gives us two requirements for a sequence model. It must accept inputs of any length with the *same* parameters, and it must be sensitive to the order of the elements. The recurrent networks of this notebook satisfy both by construction: they read the sequence one step at a time while updating a **hidden state**, a vector that summarises what has been seen so far. The same small set of weights is applied at every step, which is exactly the kind of parameter sharing across positions that made convolutions so effective across space in Notebook 06 · Convolutional Neural Networks.

### 1.1 Shapes and the taxonomy of sequence tasks

Throughout this course a batch of sequences is a tensor of shape `(batch, time, features)`. This is PyTorch's `batch_first=True` layout, which we use everywhere for readability; internally many PyTorch recurrent layers prefer `(time, batch, features)` and will transpose for you. For text, `features` is usually 1 integer token index per step (which an `nn.Embedding` then turns into a vector), for a multivariate sensor it is the number of channels, and for a spectrogram it is the number of frequency bins.

Sequence tasks are classified by the shapes of their inputs and outputs:

| Pattern | Input | Output | Examples |
|---|---|---|---|
| one-to-many | a single vector | a sequence | image captioning, generating music from a genre label |
| many-to-one | a sequence | a single vector or label | sentiment classification, activity recognition from accelerometer data |
| many-to-many (aligned) | a sequence | a sequence of the same length | part-of-speech tagging, next-token prediction (language modelling) |
| many-to-many (unaligned) | a sequence | a sequence of a different length | machine translation, speech recognition |

The same recurrent machinery serves all four: what changes is which hidden states we read outputs from and how we feed the model. Sections 5 and 8 are many-to-one problems, Section 6 is aligned many-to-many, Section 7 uses a many-to-one model repeatedly for many-step forecasting, and Section 9 builds an unaligned many-to-many model (an encoder–decoder). The next cell illustrates why a model that ignores order is not enough: it builds three sequences that contain exactly the same values in different orders.

In [ ]:
# Three toy sequences with the same "bag of values" but different orders.
t_axis = torch.linspace(0, 2 * math.pi, 40)
rising = torch.sin(t_axis / 4)                       # slowly increasing over the window
falling = torch.flip(rising, dims=[0])               # the same values, reversed in time
shuffled = rising[torch.randperm(len(t_axis))]       # the same values, in random order

X = torch.stack([rising, falling, shuffled])[..., None]   # (batch=3, time=40, features=1)
print("shape (batch, time, features):", tuple(X.shape))
print("time-averaged value per sequence :", X.mean(dim=1).squeeze(-1).cpu().numpy().round(4))
print("value at the last time step      :", X[:, -1, 0].cpu().numpy().round(4))

fig, ax = plt.subplots()
for name, seq in zip(["rising", "falling", "shuffled"], X):
    ax.plot(seq.squeeze(-1).cpu().numpy(), label=name)
ax.set_xlabel("time step"); ax.set_ylabel("value")
ax.set_title("Same values, different order")
ax.legend(); plt.tight_layout(); plt.show()

Any model that summarises a sequence by an order-invariant statistic, such as the mean over time, sees three identical inputs here and cannot tell "rising" from "falling". The last value alone does distinguish them in this toy case, but that is a coincidence of the example, not a modelling strategy: in general the information we need is spread across the sequence and only its *arrangement* tells the classes apart. We need a model whose internal state depends on the history *and its order*.

> **Key idea.** A sequence model reads its input one step at a time and updates a hidden state with the same parameters at every step. Because the update is applied repeatedly, the model handles any length, and because each update depends on the previous state, the model is sensitive to order.

## 2. The vanilla RNN from scratch

### 2.1 The recurrence

The simplest recurrent neural network maintains a hidden state $h_t \in \mathbb{R}^{H}$ and updates it from the previous state and the current input $x_t \in \mathbb{R}^{D}$:

$$
h_t = \tanh\!\left(W_{xh}\, x_t + W_{hh}\, h_{t-1} + b\right), \qquad h_0 = \mathbf{0}.
$$

The matrix $W_{xh} \in \mathbb{R}^{H \times D}$ maps the input into the state space, $W_{hh} \in \mathbb{R}^{H \times H}$ mixes the previous state, and $b \in \mathbb{R}^{H}$ is a bias. The hyperbolic tangent squashes the result into $(-1, 1)$, which keeps the state bounded no matter how long the sequence is. If we want an output at each step we add a **readout** $\hat y_t = W_{hy} h_t + c$; for a many-to-one task we read out only from the final state $h_T$.

Three features of this definition deserve attention.

1. **Parameter sharing across time.** The same $W_{xh}, W_{hh}, b$ are used at every step. The number of parameters is $HD + H^2 + H$ regardless of the sequence length $T$. Compare this with an MLP on the flattened sequence, whose first layer alone would need $T \cdot D \cdot H$ weights and could only ever handle one fixed $T$.
2. **Unrolling.** If we draw the computation for a specific $T$ we obtain a feed-forward network with $T$ layers that all share weights: $x_1 \to h_1 \to h_2 \to \dots \to h_T$, with $x_t$ injected at layer $t$. This "unrolled graph" is what autograd differentiates in Section 3, and it makes the vanishing-gradient discussion of Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization immediately relevant: a sequence of length 100 is a 100-layer-deep network.
3. **A state, not a memory.** $h_t$ is a fixed-size vector; it must compress everything relevant about $x_1, \dots, x_t$. Whether it can *keep* information for many steps is exactly the question that Sections 3–5 answer.

### 2.2 Implementation

We implement the cell as a small `nn.Module` with three parameters, and a helper that unrolls it over a `(batch, time, features)` tensor with a Python loop. Note that the loop is over *time*, while each step processes the whole batch at once with a matrix product. We follow PyTorch's initialisation convention for the weight matrices (uniform in $\pm 1/\sqrt{H}$; the bias starts at zero here) so that the numbers are comparable with `nn.RNN`.

In [ ]:
class RNNCellScratch(nn.Module):
    """One step of a vanilla (Elman) RNN: h_t = tanh(W_xh x_t + W_hh h_{t-1} + b)."""

    def __init__(self, input_size, hidden_size):
        super().__init__()
        k = 1.0 / math.sqrt(hidden_size)
        self.W_xh = nn.Parameter(torch.empty(hidden_size, input_size).uniform_(-k, k))
        self.W_hh = nn.Parameter(torch.empty(hidden_size, hidden_size).uniform_(-k, k))
        self.b = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x_t, h_prev):
        # x_t: (batch, input_size), h_prev: (batch, hidden_size) -> (batch, hidden_size)
        return torch.tanh(x_t @ self.W_xh.T + h_prev @ self.W_hh.T + self.b)


def rnn_forward(cell, x, h0=None):
    """Unroll `cell` over x of shape (batch, time, features).

    Returns all hidden states (batch, time, hidden) and the final state (batch, hidden)."""
    B, T, _ = x.shape
    H = cell.W_hh.shape[0]
    h = torch.zeros(B, H, device=x.device) if h0 is None else h0
    outputs = []
    for t in range(T):            # the loop is over time; each step handles the whole batch
        h = cell(x[:, t], h)
        outputs.append(h)
    return torch.stack(outputs, dim=1), h


cell = RNNCellScratch(input_size=5, hidden_size=8)
x = torch.randn(3, 7, 5)                      # batch of 3 sequences, 7 steps, 5 features
outputs, h_last = rnn_forward(cell, x)
print("outputs:", tuple(outputs.shape), "| last state:", tuple(h_last.shape))
print("parameters:", sum(p.numel() for p in cell.parameters()),
      "= H*D + H*H + H =", 8 * 5 + 8 * 8 + 8)

### 2.3 Checking against `nn.RNN`

PyTorch's `nn.RNN` implements the same recurrence (with `nonlinearity="tanh"` by default) but stores *two* bias vectors, `bias_ih` and `bias_hh`, a historical artefact of the cuDNN implementation; their sum plays the role of our single $b$. Its weights are `weight_ih_l0` (shape `(H, D)`) and `weight_hh_l0` (shape `(H, H)`), where `l0` means layer 0. We copy the reference weights into our cell and compare the outputs step for step. The differences should be at the level of float32 rounding, around $10^{-7}$.

`nn.RNN` returns two things: `output`, the hidden state at *every* time step of the *last* layer, shape `(batch, time, H)` with `batch_first=True`, and `h_n`, the final hidden state of *every* layer, shape `(num_layers * num_directions, batch, H)`. With one layer and one direction, `h_n[0]` equals `output[:, -1]`.

In [ ]:
torch.manual_seed(SEED)
D, H, B, T = 5, 8, 3, 7
x = torch.randn(B, T, D)

ref = nn.RNN(input_size=D, hidden_size=H, batch_first=True)   # nonlinearity="tanh"
cell = RNNCellScratch(D, H)
with torch.no_grad():
    cell.W_xh.copy_(ref.weight_ih_l0)
    cell.W_hh.copy_(ref.weight_hh_l0)
    cell.b.copy_(ref.bias_ih_l0 + ref.bias_hh_l0)

out_ours, h_ours = rnn_forward(cell, x)
out_ref, h_ref = ref(x)
print("output shapes     :", tuple(out_ours.shape), tuple(out_ref.shape))
print("final-state shapes:", tuple(h_ours.shape), tuple(h_ref.shape), "(layers*directions, batch, H)")
print("max |difference| in outputs    :", (out_ours - out_ref).abs().max().item())
print("max |difference| in final state:", (h_ours - h_ref[0]).abs().max().item())
print("h_n[0] equals output[:, -1]    :", torch.allclose(h_ref[0], out_ref[:, -1]))
print("nn.RNN parameters:", sum(p.numel() for p in ref.parameters()),
      "(ours:", sum(p.numel() for p in cell.parameters()), "+ one extra bias vector)")

The two implementations agree to floating-point precision, so we now understand exactly what `nn.RNN` computes. The built-in module is preferable in practice because it runs the time loop in C++ (and in cuDNN on a GPU) instead of Python, but the from-scratch version is what we will use whenever we need access to *every intermediate state* for analysis, as in the next section.

> **Common pitfall.** `nn.RNN`, `nn.LSTM` and `nn.GRU` default to `batch_first=False`, i.e. they expect `(time, batch, features)`. If you feed a `(batch, time, features)` tensor without `batch_first=True` the code runs without error but silently treats your batch dimension as time. Always pass `batch_first=True` in this course, and check `output.shape` against what you expect.

## 3. Backpropagation through time

### 3.1 The unrolled graph and the chain rule

Training an RNN is ordinary backpropagation applied to the unrolled graph; because the graph extends through time, the procedure is called **backpropagation through time** (BPTT). What makes it interesting is that the same parameters appear at every step, so the gradient with respect to $W_{hh}$ is a *sum over all time steps* of contributions that each flow backwards through *all later steps*.

Let us set up the notation. Write the pre-activation as $a_t = W_{xh} x_t + W_{hh} h_{t-1} + b$, so $h_t = \tanh(a_t)$, and suppose the loss is a sum of per-step losses $L = \sum_{t=1}^{T} L_t$, where $L_t$ depends on $h_t$ through a readout. By the chain rule, the loss at time $t$ depends on $W_{hh}$ through every state $h_k$ with $k \le t$, because $W_{hh}$ was used to compute each of them:

$$
\frac{\partial L_t}{\partial W_{hh}}
= \sum_{k=1}^{t} \frac{\partial L_t}{\partial h_t}\,
\frac{\partial h_t}{\partial h_k}\,
\frac{\partial^{+} h_k}{\partial W_{hh}},
$$

where $\partial^{+} h_k / \partial W_{hh}$ denotes the "immediate" derivative that treats $h_{k-1}$ as a constant. The middle factor is the part that matters: it transports the error signal from step $t$ back to step $k$, and it is a **product of Jacobians**, one per intermediate step,

$$
\frac{\partial h_t}{\partial h_k}
= \prod_{i=k+1}^{t} \frac{\partial h_i}{\partial h_{i-1}}
= \prod_{i=k+1}^{t} \operatorname{diag}\!\left(1 - h_i^{2}\right) W_{hh},
$$

using $\tanh'(a) = 1 - \tanh^2(a)$ (the factors are matrices, so the product is ordered: the latest step, $i = t$, stands on the left and $i = k+1$ on the right). Each factor is an $H \times H$ matrix, and there are $t - k$ of them: the gradient over a lag of 50 steps is a product of 50 matrices.

### 3.2 Why gradients vanish or explode

Take norms. Since $|1 - h_i^2| \le 1$ and the norm of a product is at most the product of norms,

$$
\left\| \frac{\partial h_t}{\partial h_k} \right\|
\;\le\; \prod_{i=k+1}^{t} \big\|\operatorname{diag}(1 - h_i^2)\big\| \, \|W_{hh}\|
\;\le\; \big(\gamma\, \sigma_{\max}(W_{hh})\big)^{\,t-k},
$$

where $\sigma_{\max}$ is the largest singular value and $\gamma = 1$ is the largest possible derivative of $\tanh$. Whenever $\sigma_{\max}(W_{hh}) < 1$ the bound decays *exponentially* in the lag $t - k$: this is the **vanishing gradient** problem, and it is a guarantee, not a tendency (Pascanu, Mikolov and Bengio, 2013). Conversely, if $W_{hh}$ has an eigenvalue of modulus greater than 1 and the states are not saturated, the product can grow exponentially, the **exploding gradient** problem. Between the two lies a narrow regime in which gradients survive for a while, and even there the $\tanh$ derivatives, which are strictly smaller than one, keep chipping away at them.

The practical consequence is asymmetric. Exploding gradients are loud: the loss jumps to `nan` or the parameters blow up, and there is a simple remedy, gradient clipping (Section 3.5). Vanishing gradients are quiet: training proceeds, the loss goes down, but the model only ever learns from the last few steps of context because the error signal from a step far in the future never reaches the parameters that would have had to act on the distant past. The model *looks* trained and is simply blind to long-range structure.

### 3.3 The algorithm

In practice nobody forms the Jacobian products explicitly. Define $\delta_t = \partial L / \partial a_t$, the error at the pre-activation of step $t$. Working backwards from $t = T$,

$$
\delta_t = \left(1 - h_t^{2}\right) \odot \left( W_{hy}^{\top} \frac{\partial L_t}{\partial \hat y_t} + W_{hh}^{\top} \delta_{t+1} \right), \qquad \delta_{T+1} = \mathbf{0},
$$

and the parameter gradients are sums of outer products over time:

$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \delta_t\, h_{t-1}^{\top}, \qquad
\frac{\partial L}{\partial W_{xh}} = \sum_{t=1}^{T} \delta_t\, x_t^{\top}, \qquad
\frac{\partial L}{\partial b} = \sum_{t=1}^{T} \delta_t .
$$

This is exactly what autograd executes when you call `loss.backward()` on an unrolled RNN, and the recursion for $\delta_t$ makes the Jacobian product visible again: each step back multiplies by $W_{hh}^{\top}$ and by $1 - h_t^2$. Every intermediate state must be kept in memory until the backward pass, so BPTT costs $O(T \cdot H)$ memory per sequence, which is why long sequences are truncated (Section 3.5).

### 3.4 Measuring the decay: gradient norm versus lag

Let us watch the phenomenon directly. We build RNN steps whose recurrent matrix is $W_{hh} = g\,Q$ with $Q$ a random orthogonal matrix, so that all singular values equal the **gain** $g$ and the theory has no slack in it. We unroll for $T$ steps keeping every $h_t$, put a random linear readout on the *last* state only, and ask autograd for $\partial L / \partial h_t$ at every $t$ with `torch.autograd.grad(loss, hs)`, which accepts intermediate tensors as inputs. Plotting the norm of that gradient against the lag $T - 1 - t$ shows how much of the error signal survives a journey of that many steps back in time. We also fit a straight line to the log-norms; its slope is the per-step growth rate of the gradient ($\log$ of the effective factor per step).

In [ ]:
def make_tanh_rnn_step(gain, input_size, hidden_size):
    """Return a step function h_t = tanh(W_xh x_t + W_hh h_{t-1}) with W_hh = gain * Q, Q orthogonal."""
    Q, _ = torch.linalg.qr(torch.randn(hidden_size, hidden_size))
    W_hh = gain * Q
    W_xh = 0.1 * torch.randn(hidden_size, input_size)

    def step(x_t, h_prev):
        return torch.tanh(x_t @ W_xh.T + h_prev @ W_hh.T)

    return step


def gradient_norm_vs_lag(step, T, input_size, hidden_size):
    """Norm of dL/dh_t for every t, where L reads out the final state only."""
    x = torch.randn(1, T, input_size)
    h = torch.zeros(1, hidden_size, requires_grad=True)   # makes the whole chain part of the graph
    hs = []
    for t in range(T):
        h = step(x[:, t], h)
        hs.append(h)
    w = torch.randn(hidden_size)
    loss = (hs[-1] * w).sum()
    grads = torch.autograd.grad(loss, hs)                # gradient w.r.t. every intermediate state
    lags = np.arange(T - 1, -1, -1)                       # hs[0] is T-1 steps before the loss
    norms = np.array([g.norm().item() for g in grads])
    return lags, norms


torch.manual_seed(SEED)
T_LAG, D_LAG, H_LAG = 60, 8, 32
fig, ax = plt.subplots()
for gain in [0.5, 1.0, 1.5]:
    torch.manual_seed(SEED)
    lags, norms = gradient_norm_vs_lag(make_tanh_rnn_step(gain, D_LAG, H_LAG), T_LAG, D_LAG, H_LAG)
    slope = np.polyfit(lags, np.log(norms + 1e-300), 1)[0]
    print(f"gain {gain:.1f}: gradient norm at lag 0 = {norms[-1]:.2e}, at lag {T_LAG-1} = {norms[0]:.2e}, "
          f"fitted slope = {slope:+.3f} per step (factor {math.exp(slope):.3f})")
    ax.semilogy(lags, norms, label=f"tanh RNN, gain {gain}")
ax.set_xlabel("lag (steps between the state and the loss)")
ax.set_ylabel("gradient norm  ||dL/dh_t||")
ax.set_title("Gradient norm versus lag in a vanilla RNN")
ax.legend(); plt.tight_layout(); plt.show()

The picture matches the theory. With gain $0.5$ every step back multiplies the gradient by roughly one half, so after a few dozen steps it is numerically zero: the fitted slope is close to $\log 0.5 \approx -0.69$. With gain $1.0$, the orthogonal matrix alone would preserve the norm exactly, and what remains is the effect of the $\tanh$ derivatives, which are strictly below one whenever the state is not zero; the gradient still decays, only more slowly. With gain $1.5$ the linear part amplifies by 1.5 per step, and the curve typically rises for the first steps before the states saturate and the shrinking $\tanh$ derivatives fight back; the slope you observe on your run summarises which effect wins. Whatever the details, note the *scale* of the vertical axis: over a lag of 50 steps the ratio between the best and worst case is many orders of magnitude, and nothing in training controls which case we are in, because $W_{hh}$ changes at every optimizer step.

> **Try it.** Replace `torch.tanh` in `make_tanh_rnn_step` by the identity. The gradient norm then follows $g^{\text{lag}}$ exactly, and the gain-1 curve becomes flat: with an orthogonal recurrence and no non-linearity nothing vanishes. This observation is the basis of "orthogonal/unitary RNNs", an alternative research line to the gates of Section 4.

### 3.5 Truncated BPTT and gradient clipping

Two engineering answers to the two problems.

**Truncated BPTT** addresses cost, not the vanishing itself. Unrolling a language model over an entire book would require storing every hidden state of the book for the backward pass. Instead we cut the stream into chunks of $T_{\text{bptt}}$ steps (typically 32–256). Within a chunk we backpropagate normally; between chunks we *carry the hidden state forward* as the initial state of the next chunk but **detach** it from the graph, so that gradients do not flow across the boundary. The forward pass thus has unlimited memory (the state was computed from the whole history), while the backward pass sees at most $T_{\text{bptt}}$ steps. The model can learn dependencies longer than the chunk only indirectly, by learning to keep useful information in the state. Section 6 implements this.

**Gradient clipping** addresses exploding gradients. Before the optimizer step we compute the global norm of the gradient over all parameters, $\|g\| = \sqrt{\sum_i \|g_i\|^2}$, and if it exceeds a threshold $\tau$ we rescale the whole gradient to have norm $\tau$:

$$
g \leftarrow g \cdot \min\!\left(1, \frac{\tau}{\|g\|}\right).
$$

Clipping by norm preserves the *direction* of the update and only bounds its size, which is why it is preferred over clipping each coordinate to $[-\tau, \tau]$ (`clip_grad_value_`), which changes the direction. Typical thresholds are between 0.25 and 5; for recurrent models a value around 1 is a safe default. The next cell implements clipping by norm manually and verifies that it agrees with `torch.nn.utils.clip_grad_norm_`, which returns the *pre-clipping* norm so you can log it.

In [ ]:
def clip_grad_norm_manual(parameters, max_norm):
    """Rescale gradients in place so that their global L2 norm is at most max_norm.

    Returns the norm before clipping (like torch.nn.utils.clip_grad_norm_)."""
    params = [p for p in parameters if p.grad is not None]
    total = torch.norm(torch.stack([p.grad.norm() for p in params]))
    coef = max_norm / (total + 1e-6)
    if coef < 1.0:
        for p in params:
            p.grad.mul_(coef)
    return total


torch.manual_seed(SEED)
probe = nn.Sequential(nn.Linear(5, 32), nn.Tanh(), nn.Linear(32, 1))
x_probe, y_probe = torch.randn(64, 5), torch.randn(64, 1)

def gradients_of(model):
    model.zero_grad(set_to_none=True)
    (200 * F.mse_loss(model(x_probe), y_probe)).backward()   # scaled up to force a large gradient

gradients_of(probe)
norm_manual = clip_grad_norm_manual(probe.parameters(), max_norm=1.0)
clipped_manual = [p.grad.clone() for p in probe.parameters()]

gradients_of(probe)
norm_torch = nn.utils.clip_grad_norm_(probe.parameters(), max_norm=1.0)
clipped_torch = [p.grad.clone() for p in probe.parameters()]

print(f"norm before clipping: manual {norm_manual:.4f} | torch {norm_torch:.4f}")
print(f"norm after clipping : {torch.norm(torch.stack([g.norm() for g in clipped_manual])):.4f}")
print("max |difference| between the two implementations:",
      max((a - b).abs().max().item() for a, b in zip(clipped_manual, clipped_torch)))

The pre-clipping norm is far above the threshold, the post-clipping norm is exactly 1, and our implementation agrees with PyTorch's. In every training loop of this notebook you will see the line `nn.utils.clip_grad_norm_(model.parameters(), 1.0)` between `backward()` and `step()`; for recurrent models it is not optional polish but the difference between a run that trains and one that produces `nan` after a rare exploding batch.

> **Going deeper.** Clipping interacts with the optimizer. With plain SGD the clipped gradient is the update, so the threshold directly caps the step length. With Adam the update is normalised by the running second moment, so a single clipped gradient has a much smaller effect on that step, but clipping still protects the *moment estimates* from being polluted by one enormous value, which would otherwise depress the effective learning rate for hundreds of subsequent steps.

## 4. Gated architectures: LSTM and GRU

### 4.1 The idea: an additive memory with learned gates

Section 3 showed that the vanilla RNN *overwrites* its state at every step, $h_t = \tanh(\cdot)$, and that the gradient flowing back through such an overwrite is multiplied by a matrix whose size the network does not control. The **Long Short-Term Memory** network (Hochreiter and Schmidhuber, 1997) changes the recurrence in one crucial way: it adds a second state, the **cell state** $c_t$, which is updated *additively*,

$$
c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t ,
$$

where $f_t, i_t \in (0,1)^H$ are **gates**, vectors of sigmoids computed from the current input and the previous hidden state, and $\tilde c_t$ is a candidate value. If the forget gate $f_t$ is close to one and the input gate $i_t$ close to zero, the cell simply copies itself, $c_t \approx c_{t-1}$, and the gradient $\partial c_t / \partial c_{t-1} \approx I$ passes through unchanged. Hochreiter called this the *constant error carousel*. The network now *decides*, per coordinate and per step, whether to keep, replace, or blend its memory, and the decay rate of the gradient is a learned quantity instead of a side effect of $W_{hh}$ and $\tanh'$.

### 4.2 The LSTM cell equations

With $x_t \in \mathbb{R}^D$ and the previous hidden state $h_{t-1} \in \mathbb{R}^H$, one step of an LSTM cell computes

$$
\begin{aligned}
i_t &= \sigma\!\left(W_{ii} x_t + b_{ii} + W_{hi} h_{t-1} + b_{hi}\right) && \text{input gate: how much of the candidate to write} \\
f_t &= \sigma\!\left(W_{if} x_t + b_{if} + W_{hf} h_{t-1} + b_{hf}\right) && \text{forget gate: how much of the old cell to keep} \\
\tilde c_t &= \tanh\!\left(W_{ig} x_t + b_{ig} + W_{hg} h_{t-1} + b_{hg}\right) && \text{candidate cell value} \\
o_t &= \sigma\!\left(W_{io} x_t + b_{io} + W_{ho} h_{t-1} + b_{ho}\right) && \text{output gate: how much of the cell to expose} \\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde c_t && \text{additive memory update} \\
h_t &= o_t \odot \tanh(c_t) && \text{hidden state (also the output)}
\end{aligned}
$$

The four affine maps are stacked into two matrices, $W_{ih} \in \mathbb{R}^{4H \times D}$ and $W_{hh} \in \mathbb{R}^{4H \times H}$, in the order (input, forget, candidate, output). This is precisely PyTorch's parametrisation in `nn.LSTMCell` and `nn.LSTM`: a single matrix product produces all $4H$ pre-activations, which are then split into four chunks. The parameter count is $4(HD + H^2 + 2H)$, four times that of a vanilla RNN of the same width.

Two remarks about the design. The hidden state $h_t$ is a *gated, squashed view* of the memory: the cell can hold a value for a hundred steps while $h_t$ (and hence the output) ignores it, because $o_t$ decides what to expose. And the candidate uses $\tanh$ while all gates use the sigmoid, because gates must be in $[0,1]$ to act as soft switches while the content must be able to take either sign.

### 4.3 Implementation from scratch

We store the four stacked weight matrices exactly as PyTorch does, so that we can later copy weights from `nn.LSTMCell` and verify the two agree.

In [ ]:
class LSTMCellScratch(nn.Module):
    """One LSTM step with PyTorch's parametrisation (gates stacked in the order i, f, g, o)."""

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        k = 1.0 / math.sqrt(hidden_size)
        self.W_ih = nn.Parameter(torch.empty(4 * hidden_size, input_size).uniform_(-k, k))
        self.W_hh = nn.Parameter(torch.empty(4 * hidden_size, hidden_size).uniform_(-k, k))
        self.b_ih = nn.Parameter(torch.zeros(4 * hidden_size))
        self.b_hh = nn.Parameter(torch.zeros(4 * hidden_size))

    def forward(self, x_t, state):
        h_prev, c_prev = state
        gates = x_t @ self.W_ih.T + self.b_ih + h_prev @ self.W_hh.T + self.b_hh   # (batch, 4H)
        i, f, g, o = gates.chunk(4, dim=-1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c_prev + i * g            # additive memory update
        h = o * torch.tanh(c)             # gated exposure of the memory
        return h, c


def lstm_forward(cell, x, state=None):
    """Unroll an LSTM cell over x of shape (batch, time, features); returns (outputs, (h, c))."""
    B, T, _ = x.shape
    H = cell.hidden_size
    if state is None:
        state = (torch.zeros(B, H, device=x.device), torch.zeros(B, H, device=x.device))
    outputs = []
    for t in range(T):
        h, c = cell(x[:, t], state)
        state = (h, c)
        outputs.append(h)
    return torch.stack(outputs, dim=1), state


lstm_cell = LSTMCellScratch(input_size=6, hidden_size=10)
print("parameters:", sum(p.numel() for p in lstm_cell.parameters()),
      "= 4*(H*D + H*H + 2H) =", 4 * (10 * 6 + 10 * 10 + 2 * 10))

### 4.4 Verification against `nn.LSTMCell`

`nn.LSTMCell(input_size, hidden_size)` takes one input step `x_t` of shape `(batch, D)` and a state tuple `(h, c)`, and returns the new tuple. We copy its four parameter tensors into our cell and run both over a random sequence of twelve steps, feeding each its own state, and record the largest deviation in either state.

In [ ]:
torch.manual_seed(SEED)
D, H, B, T = 6, 10, 4, 12
ref_cell = nn.LSTMCell(D, H)
ours = LSTMCellScratch(D, H)
with torch.no_grad():
    ours.W_ih.copy_(ref_cell.weight_ih); ours.W_hh.copy_(ref_cell.weight_hh)
    ours.b_ih.copy_(ref_cell.bias_ih);   ours.b_hh.copy_(ref_cell.bias_hh)

x = torch.randn(B, T, D)
h_a = c_a = torch.zeros(B, H)
h_b = c_b = torch.zeros(B, H)
max_diff = 0.0
for t in range(T):
    h_a, c_a = ours(x[:, t], (h_a, c_a))
    h_b, c_b = ref_cell(x[:, t], (h_b, c_b))
    max_diff = max(max_diff, (h_a - h_b).abs().max().item(), (c_a - c_b).abs().max().item())
print(f"max |difference| over {T} steps (h and c): {max_diff:.2e}")

# The multi-layer nn.LSTM stores the same tensors per layer under the names weight_ih_l0, ...
ref_lstm = nn.LSTM(D, H, batch_first=True)
with torch.no_grad():
    for name in ["weight_ih", "weight_hh", "bias_ih", "bias_hh"]:
        getattr(ref_lstm, name + "_l0").copy_(getattr(ref_cell, name))
out_ref, (h_ref, c_ref) = ref_lstm(x)
out_ours, (h_ours, c_ours) = lstm_forward(ours, x)
print(f"nn.LSTM vs from-scratch unrolled: max |difference| = {(out_ref - out_ours).abs().max().item():.2e}")

The from-scratch cell reproduces `nn.LSTMCell` and the unrolled `nn.LSTM` to rounding error. As with the RNN, we will use the built-in modules for training, and the scratch version when we need every intermediate state.

### 4.5 Why the additive cell state preserves gradients

Differentiate the memory update with respect to the previous cell state. Holding the gates fixed (their own dependence on $h_{t-1}$ adds further terms, which are usually small when the gates are saturated),

$$
\frac{\partial c_t}{\partial c_{t-1}} = \operatorname{diag}(f_t) \;+\; (\text{terms through } h_{t-1} = o_{t-1} \odot \tanh(c_{t-1})).
$$

The dominant factor is a *diagonal* matrix with entries in $(0, 1)$ that the network sets itself, instead of the fixed product $\operatorname{diag}(1 - h_t^2)\, W_{hh}$ of the vanilla RNN. The gradient transported over a lag of $n$ steps along the cell path therefore scales like $\prod f$, and for a coordinate whose forget gate sits at $f \approx 0.99$ the signal is still at $0.99^{100} \approx 0.37$ after a hundred steps. Nothing forces the gates to be near one, but they *can* be, and training will push them there for whichever memory coordinates need long retention. This is a qualitative difference: the vanilla RNN has no parameter setting that makes long-lag gradients survive without also risking explosion.

A well-known practical consequence is the **forget-gate bias initialisation**: setting $b_{f}$ to 1 or 2 at initialisation (so $f \approx 0.73$–$0.88$ before any learning) makes early training much easier on long-range tasks, because the network starts out remembering instead of starting out forgetting (Jozefowicz, Zaremba and Sutskever, 2015). PyTorch initialises all biases uniformly around zero, so $f \approx 0.5$ by default; we will set the forget bias ourselves.

The next cell repeats the gradient-versus-lag experiment of Section 3.4 with our LSTM cell, for three values of the forget-gate bias, and measures the gradient of the final readout with respect to every intermediate *cell state* $c_t$. The dashed reference curve is the vanilla RNN at gain 1.

In [ ]:
def lstm_gradient_norm_vs_lag(forget_bias, T, input_size, hidden_size):
    """Norm of dL/dc_t for every t in an LSTM whose forget-gate bias is set to `forget_bias`."""
    cell = LSTMCellScratch(input_size, hidden_size)
    with torch.no_grad():
        cell.b_ih[hidden_size:2 * hidden_size] = forget_bias   # forget-gate slice of the stacked bias
        cell.b_hh[hidden_size:2 * hidden_size] = 0.0
    x = torch.randn(1, T, input_size)
    h = torch.zeros(1, hidden_size)
    c = torch.zeros(1, hidden_size)
    cs = []
    for t in range(T):
        h, c = cell(x[:, t], (h, c))
        cs.append(c)
    w = torch.randn(hidden_size)
    loss = (h * w).sum()
    grads = torch.autograd.grad(loss, cs)
    lags = np.arange(T - 1, -1, -1)
    norms = np.array([g.norm().item() for g in grads])
    return lags, norms


fig, ax = plt.subplots()
torch.manual_seed(SEED)
lags, norms = gradient_norm_vs_lag(make_tanh_rnn_step(1.0, D_LAG, H_LAG), T_LAG, D_LAG, H_LAG)
ax.semilogy(lags, norms, "k--", label="vanilla RNN, gain 1.0")
for fb in [0.0, 2.0, 5.0]:
    torch.manual_seed(SEED)
    lags, norms = lstm_gradient_norm_vs_lag(fb, T_LAG, D_LAG, H_LAG)
    slope = np.polyfit(lags, np.log(norms + 1e-300), 1)[0]
    print(f"forget bias {fb:.0f} (f ~ {torch.sigmoid(torch.tensor(fb)).item():.3f}): "
          f"fitted slope {slope:+.3f} per step, factor {math.exp(slope):.3f}, "
          f"norm at lag {T_LAG-1}: {norms[0]:.2e}")
    ax.semilogy(lags, norms, label=f"LSTM, forget bias {fb:.0f}")
ax.set_xlabel("lag (steps between the state and the loss)")
ax.set_ylabel("gradient norm  ||dL/dc_t||")
ax.set_title("Gradient norm versus lag: LSTM cell path")
ax.legend(); plt.tight_layout(); plt.show()

The measured per-step factor tracks the value of the forget gate: with the bias at zero the gate sits near $0.5$ and the gradient loses roughly a third to a half of its norm at every step, which is no better than the vanilla-RNN reference curve; with the bias at 2 the gate is at $0.88$ and the measured factor is close to it or slightly above (the paths through $h$ add to the cell path); with the bias at 5 the gate is at $0.993$ and the gradient arrives essentially undiminished after sixty steps, and may even grow a little. The important point is not that any particular bias is "right", but that the *rate* is now a function of learnable quantities. During training the LSTM can open the forget gate for the coordinates that need to carry information across long spans and close it for those that should track the recent past. The vanilla RNN has a single, global rate set by $W_{hh}$ and the saturation of $\tanh$.

> **Common pitfall.** The gradient argument concerns the *cell* path. The hidden state $h_t = o_t \odot \tanh(c_t)$ still passes through a squashing non-linearity, and gradients that travel through $h$ (into the gates of the next step) are subject to the same shrinkage as in a vanilla RNN. LSTMs are much better at long dependencies, not immune to the problem; in practice they reliably learn dependencies over a few hundred steps, not thousands. Transformers (Notebook 09 · Attention and Transformers) sidestep the issue entirely by connecting every position to every other directly.

### 4.6 The GRU

The **Gated Recurrent Unit** (Cho et al., 2014) keeps the additive-update idea but merges the cell and hidden states and uses two gates instead of three:

$$
\begin{aligned}
r_t &= \sigma\!\left(W_{ir} x_t + b_{ir} + W_{hr} h_{t-1} + b_{hr}\right) && \text{reset gate} \\
z_t &= \sigma\!\left(W_{iz} x_t + b_{iz} + W_{hz} h_{t-1} + b_{hz}\right) && \text{update gate} \\
n_t &= \tanh\!\left(W_{in} x_t + b_{in} + r_t \odot \left(W_{hn} h_{t-1} + b_{hn}\right)\right) && \text{candidate state} \\
h_t &= (1 - z_t) \odot n_t + z_t \odot h_{t-1} && \text{interpolation}
\end{aligned}
$$

The update gate $z_t$ plays the roles of both the forget gate and the input gate, coupled so that they sum to one: whatever fraction of the old state is kept, the complementary fraction is written from the candidate. The reset gate $r_t$ decides how much of the previous state feeds into the candidate, which lets the unit ignore its history when a new segment begins. With no output gate and no separate cell, a GRU has $3(HD + H^2 + 2H)$ parameters, three quarters of an LSTM of the same width. Note PyTorch's specific placement of $r_t$ *inside* the $\tanh$, multiplying $W_{hn} h_{t-1} + b_{hn}$; other presentations apply $r_t$ to $h_{t-1}$ before the matrix product, which is a slightly different model. You will implement this cell and verify it against `nn.GRUCell` in Exercise 1.

Which should you use? Empirically the two are close; the GRU is a little cheaper and often converges faster on small datasets, while the LSTM's separate cell gives it an edge on tasks that need to store information for a long time while producing unrelated outputs (Chung et al., 2014; Greff et al., 2017). Both remain useful today for small and medium sequence problems, streaming inference on devices, and as components inside larger systems; for large-scale language modelling the Transformer has replaced them.

### 4.7 The `nn.LSTM` API

The built-in `nn.LSTM` (and likewise `nn.GRU`, `nn.RNN`) unrolls the cell over a whole sequence, optionally stacks several layers (`num_layers`), and optionally runs a second copy backwards over time (`bidirectional=True`) whose outputs are concatenated with the forward ones. The state tensors have shape `(num_layers * num_directions, batch, H)`, laid out with the direction index varying fastest: entry `layer * num_directions + direction`. The output tensor has shape `(batch, time, num_directions * H)` and contains only the *last* layer's states. The cell below checks these conventions, including the subtle fact that the backward direction's *final* state corresponds to time step 0.

In [ ]:
torch.manual_seed(SEED)
D, H, L, B, T = 8, 16, 2, 3, 5
lstm = nn.LSTM(input_size=D, hidden_size=H, num_layers=L, batch_first=True, bidirectional=True)
x = torch.randn(B, T, D)
out, (h_n, c_n) = lstm(x)                     # initial state defaults to zeros
print("out :", tuple(out.shape), "= (batch, time, directions*H)")
print("h_n :", tuple(h_n.shape), "= (layers*directions, batch, H); c_n has the same shape")

h_view = h_n.view(L, 2, B, H)                 # (layer, direction, batch, H)
print("forward  final state of last layer == out[:, -1, :H] :", torch.allclose(h_view[-1, 0], out[:, -1, :H]))
print("backward final state of last layer == out[:,  0, H:] :", torch.allclose(h_view[-1, 1], out[:, 0, H:]))

# Passing an explicit initial state, e.g. to continue a sequence where the last call stopped:
out2, (h_n2, c_n2) = lstm(x, (h_n, c_n))
print("continuing from a given state works:", tuple(out2.shape))


def set_forget_bias(rnn_module, value=1.0):
    """Bias the 'keep the state' gate of every layer/direction to `value` (Section 4.5).

    For an nn.LSTM this is the forget gate, for an nn.GRU the update gate z; in both modules that gate
    is the second chunk of the stacked gate biases (order i, f, g, o for the LSTM; r, z, n for the GRU)."""
    with torch.no_grad():
        Hh = rnn_module.hidden_size
        for name, p in rnn_module.named_parameters():
            if name.startswith("bias_ih"):
                p[Hh:2 * Hh] = value
            elif name.startswith("bias_hh"):
                p[Hh:2 * Hh] = 0.0


set_forget_bias(lstm, 1.0)
print("forget-gate bias of layer 0, forward:", lstm.bias_ih_l0[H:2 * H][:4].tolist(), "...")

Two further options matter in practice. `dropout=p` applies dropout *between* stacked layers (not inside the recurrence), and only has an effect when `num_layers > 1` (Exercise 4 measures it). And when sequences in a batch have different lengths, the input should be a `PackedSequence` produced by `pack_padded_sequence`, so that the recurrence stops at each sequence's true end instead of running on into the padding; Section 8 shows why this matters and how to do it.

## 5. Experiment: learning long-range dependencies

Theory says the vanilla RNN cannot transport gradients across long lags while gated cells can. Let us test the claim with a controlled task in which the *only* thing that matters is information from far in the past.

### 5.1 The delayed-recall task

Each sequence has length $T$. The first token is a **key** drawn from $K = 4$ classes; the remaining $T - 1$ tokens are **distractors** drawn from a disjoint set of 8 symbols and carry no information. The target is the key. A model reads the whole sequence and predicts the class from its final hidden state, so it must carry the identity of the first token across $T - 1$ irrelevant steps. Chance accuracy is 25%. By varying $T$ we measure how far each architecture can carry one nugget of information.

We use the same architecture for the three contestants: an embedding of size 16, one recurrent layer of width 32 (`nn.RNN`, `nn.LSTM` or `nn.GRU`), and a linear classifier on the final state. All three train with Adam, gradient clipping at norm 1, and the same number of steps; the LSTM gets a forget-gate bias of 2 ($f \approx 0.88$ at initialisation, the upper end of the range recommended in Section 4.5), and the GRU the analogous update-gate bias, so that both gated cells start out remembering. We re-seed before every run so that the comparison is fair.

In [ ]:
N_CLASSES, N_DISTRACT = 4, 8
VOCAB_RECALL = N_CLASSES + N_DISTRACT

def make_recall_batch(batch, T):
    """Sequences of length T: position 0 holds the key (0..3), the rest are distractors (4..11)."""
    key = torch.randint(0, N_CLASSES, (batch,))
    noise = torch.randint(N_CLASSES, VOCAB_RECALL, (batch, T - 1))
    x = torch.cat([key[:, None], noise], dim=1)
    return x, key


class RecallModel(nn.Module):
    """Embedding -> one recurrent layer -> linear head on the final hidden state (many-to-one)."""

    def __init__(self, kind, vocab_size, n_classes, emb_dim=16, hidden_size=32):
        super().__init__()
        rnn_cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[kind]
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.rnn = rnn_cls(emb_dim, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, n_classes)

    def forward(self, x):
        out, _ = self.rnn(self.emb(x))     # out: (batch, time, hidden)
        return self.head(out[:, -1])       # read out from the final state


def train_recall(kind, T, steps, lr=5e-3, batch_size=64, n_test=1000, clip="norm", optimizer="adam", gate_bias=2.0):
    """Train one model on the delayed-recall task of length T and return its test accuracy."""
    torch.manual_seed(SEED)
    model = RecallModel(kind, VOCAB_RECALL, N_CLASSES).to(device)
    if kind in ("LSTM", "GRU"):
        set_forget_bias(model.rnn, gate_bias)    # forget gate (LSTM) / update gate (GRU) start near 0.88
    if optimizer == "adam":
        opt = torch.optim.Adam(model.parameters(), lr=lr)
    else:
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    model.train()
    for step in range(steps):
        x, y = make_recall_batch(batch_size, T)
        x, y = x.to(device), y.to(device)
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        if clip == "norm":
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        elif clip == "value":
            nn.utils.clip_grad_value_(model.parameters(), 0.1)
        opt.step()
    model.eval()
    with torch.no_grad():
        x, y = make_recall_batch(n_test, T)
        acc = (model(x.to(device)).argmax(dim=-1) == y.to(device)).float().mean().item()
    return acc


x_demo, y_demo = make_recall_batch(2, 8)
print("example sequences:", x_demo.tolist(), "-> keys:", y_demo.tolist())

### 5.2 Accuracy versus sequence length

Under `FAST_RUN` we train for 300 steps at three lengths; the full run uses six lengths up to 160 and many more steps. Each run takes a few seconds on a CPU.

In [ ]:
LENGTHS = [5, 15, 30] if FAST_RUN else [5, 10, 20, 40, 80, 160]
STEPS_RECALL = 300 if FAST_RUN else 2000

results = {kind: [] for kind in ["RNN", "LSTM", "GRU"]}
t0 = time.time()
for T in LENGTHS:
    for kind in results:
        acc = train_recall(kind, T, STEPS_RECALL)
        results[kind].append(acc)
    print(f"T = {T:4d} | " + " | ".join(f"{k}: {results[k][-1]:.3f}" for k in results)
          + f" | {time.time() - t0:5.1f}s")

fig, ax = plt.subplots()
for kind, accs in results.items():
    ax.plot(LENGTHS, accs, marker="o", label=kind)
ax.axhline(1 / N_CLASSES, color="gray", linestyle=":", label="chance")
ax.set_xscale("log"); ax.set_xlabel("sequence length T (log scale)"); ax.set_ylabel("test accuracy")
ax.set_ylim(0, 1.05); ax.set_title(f"Delayed recall: accuracy vs. length ({STEPS_RECALL} steps per run)")
ax.legend(); plt.tight_layout(); plt.show()

The pattern that emerges is the one the gradient analysis predicts. At short lengths every architecture solves the task. As the lag grows the vanilla RNN's accuracy collapses towards chance: not because a 32-dimensional state cannot *hold* two bits of information for fifty steps (it trivially could), but because the gradient that would teach it to do so has vanished by the time it reaches the first step. The LSTM and GRU keep learning the task at lengths where the RNN has given up, and where they do eventually fail (at the longest lengths of the full run, or when the step budget is too small), it is a matter of optimisation time rather than a hard wall: give them more steps and they usually get there. If a gated curve has not reached 100% at a length where you expected it to, increase `STEPS_RECALL` and rerun that length; with a fixed budget the outcome at the border of feasibility has some variance from run to run.

> **Try it.** Rerun the LSTM at the longest `FAST_RUN` length with `gate_bias=1.0` (and then `0.0`). With the gate at $0.73$ instead of $0.88$ the LSTM typically *fails* to learn $T = 30$ within this budget, exactly as the gradient-versus-lag plot of Section 4.5 predicts: $0.73^{30} \approx 10^{-4}$ of the error signal survives the journey back to the first step, against $0.88^{30} \approx 0.02$. Give it a few thousand steps and it recovers, since the gate is learnable; the bias only decides how hard the start is.

## 6. A character-level language model on Tiny Shakespeare

### 6.1 What a language model is

A **language model** assigns a probability to a sequence of tokens. By the chain rule of probability any joint distribution over a sequence factorises into a product of conditionals,

$$
p(x_1, x_2, \dots, x_T) = \prod_{t=1}^{T} p\!\left(x_t \mid x_1, \dots, x_{t-1}\right),
$$

so it suffices to model "the next token given everything so far". That is an *aligned many-to-many* task: at every position the network outputs a categorical distribution over the vocabulary, and the target at every position is simply the next input token (the target sequence is the input sequence advanced by one step). Training maximises the log-likelihood of the data, i.e. minimises the average cross-entropy between the model's next-token distribution and the token that actually came next. The exact same objective, scaled up by many orders of magnitude in data and compute, trains the large language models of Notebook 09 · Attention and Transformers and Notebook 10 · NLP with Pretrained Transformers (Hugging Face).

We work at the level of **characters**: the vocabulary is the set of distinct characters in the corpus (about 65 for Tiny Shakespeare), each character is one token, and the model must learn spelling, punctuation, the play-script layout, and eventually words and phrases, entirely from statistics. Character models are a wonderful teaching device because the vocabulary is tiny, no tokenizer is needed, and the samples make the model's progress visible: first random characters, then plausible letter combinations, then real words, then lines that scan. Subword tokenisation, the practical middle ground between characters and words, is the subject of Notebook 10 · NLP with Pretrained Transformers (Hugging Face).

### 6.2 Data

The corpus is Tiny Shakespeare, about one million characters of concatenated plays, made popular by Andrej Karpathy's 2015 essay on character RNNs. The cell downloads it once into `DATA_DIR`; if the download fails (no network), it falls back to a short built-in play-style text written for this course so that every cell still runs. The fallback is only a few thousand characters, which is enough to exercise the code but far too little to learn English from; if you see the fallback message, expect the samples to be poor and try again with a connection.

In [ ]:
FALLBACK_TEXT = """FIRST CITIZEN:
Before we go one step further, hear me speak.
The granaries are full, and yet we starve;
The gates are shut against us, and the guard
That should be ours is paid to keep us out.

SECOND CITIZEN:
Speak on, for hunger makes a patient listener.

FIRST CITIZEN:
Then hear this: he that holds the keys tonight
Holds all our children's suppers in his fist.
Shall we go begging to a bolted door,
Or shall we make the door remember us?

ALL:
Let it remember! To the gates, to the gates!

SECOND CITIZEN:
Soft, soft. A crowd is brave until it meets
A single man who will not step aside.
What would you have us do when we are there?

FIRST CITIZEN:
Ask, first. Then ask again with louder voices.
And if the third request should find no ear,
The fourth shall need no words.

Enter a STEWARD, with torches.

STEWARD:
Good people, what is this? The hour is late,
The lord is sleeping, and the streets are cold.
Go home and warm yourselves; the morning brings
A fairer time for grievances than night.

FIRST CITIZEN:
The morning brings the same shut gate, good steward,
And one more empty bowl upon the table.
We are not here to quarrel with the dark;
We are here because the dark has come indoors.

STEWARD:
I am a servant, not the master of the store.
What I am told to keep, I keep; what I am told
To open, that I open. Bring me his command
And I will fling the doors so wide
The very rats will think the harvest has come twice.

SECOND CITIZEN:
Then fetch him, and be quick; we shall be counting.

STEWARD:
He will not come for counting. He will come
For fire, or not at all.

FIRST CITIZEN:
You heard him, friends. He asks us for a fire.

SECOND CITIZEN:
He asked for no such thing; he named the cost
Of being heard, and hoped we could not pay it.

FIRST CITIZEN:
Then let us pay in coin he did not name.
Sit down, all of you. Sit down upon the stones
Before his door, and let no wagon pass,
No cart, no horse, no message and no bread,
Until the keys come down. We need no torches;
Our patience is the only flame we bring,
And it will burn the longer for being cold.

ALL:
Sit! Sit! Let no wagon pass!

LORD HALDEN:
What noise is this, that will not be a noise?
I looked for shouting, and I find a silence
That has more teeth in it than any shout.
Steward! What do they want?

STEWARD:
They want the gates, my lord, and they will wait.

LORD HALDEN:
Wait? Let them wait; the night is theirs to waste.

STEWARD:
The night is theirs, my lord, and so is the road.
Nothing has come in since they sat down,
And nothing has gone out. Your cousin's grain
Stands in three carts a mile beyond the bridge
And cannot pass. The buyer from the coast
Has turned his mule around.

LORD HALDEN:
So. They have found the one thing that I fear,
Which is not fire, but the cost of stillness.
Well then. A man who counts must sometimes count
What he would rather not. Give them the keys,
But say it was my mercy, not their sitting.

STEWARD:
They will not care what it is called, my lord,
So long as it is bread.

LORD HALDEN:
No. And that is why they will be back.

FIRST CITIZEN:
The keys! The keys come down! Rise up, and gently;
We came as neighbours, and we leave as such.
Take what you need and mark what you have taken,
For we will pay it back when harvest comes,
And show him that a debt can be a bridge.

SECOND CITIZEN:
And if he burns the bridge?

FIRST CITIZEN:
Then we have learned to sit, and can sit twice.

STEWARD:
I have served twenty years behind that gate
And never once was asked what I would open.
Tonight a crowd of strangers asked me nothing,
And sat, and by their sitting turned the key
That all my service never dared to touch.
What is a lock, if not a kind of promise
That someone on the other side will listen?
And what is service, if the promise breaks?
I think I shall go home by the long road,
And count the carts, and count the empty bowls,
And learn which of the two I am afraid of.
"""


def load_tiny_shakespeare(data_dir, min_chars=100_000):
    """Download Tiny Shakespeare into data_dir once; fall back to the built-in text when offline."""
    path = data_dir / "tinyshakespeare.txt"
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    if not path.exists():
        try:
            with urllib.request.urlopen(url, timeout=30) as response:
                raw = response.read().decode("utf-8")
            path.write_text(raw, encoding="utf-8")
            print(f"Downloaded Tiny Shakespeare to {path} ({len(raw):,} characters).")
        except Exception as exc:
            print(f"Could not download Tiny Shakespeare ({type(exc).__name__}: {exc}).")
            print("Using the built-in fallback text instead, so that the notebook still runs offline.")
    if path.exists() and path.stat().st_size >= min_chars:
        return path.read_text(encoding="utf-8"), "Tiny Shakespeare"
    return FALLBACK_TEXT, "built-in fallback text"


text, text_source = load_tiny_shakespeare(DATA_DIR)
print(f"corpus: {text_source} | {len(text):,} characters | {len(set(text))} distinct characters")
print("first 200 characters:\n" + text[:200])

### 6.3 Vocabulary, encoding and contiguous streams

The tokenizer for a character model is two dictionaries: `stoi` maps each character to an integer index and `itos` maps back. The whole corpus becomes one long `LongTensor` of indices, of which we hold out the last 10% as validation data (a *contiguous* split, never a random one, because adjacent characters are almost identical as training examples and a random split would leak).

The batching scheme deserves care, because it is what makes truncated BPTT work. We cut the training tensor into `BATCH` equal **streams** by reshaping it to `(BATCH, n)`: stream 0 is the first `n` characters of the corpus, stream 1 the next `n`, and so on. A mini-batch then consists of the *same* column range `[k·T, (k+1)·T)` of every stream, so consecutive mini-batches continue each stream exactly where the previous one stopped. The target for a chunk is the window that starts one character later in the stream, `[k·T + 1, (k+1)·T + 1)`, so that the target at every position is the next character.

In [ ]:
chars = sorted(set(text))
VOCAB_SIZE = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return torch.tensor([stoi[ch] for ch in s], dtype=torch.long)

def decode(ids):
    return "".join(itos[int(i)] for i in ids)

ids = encode(text)
n_split = int(0.9 * len(ids))
train_ids, val_ids = ids[:n_split], ids[n_split:]
print(f"vocabulary size {VOCAB_SIZE} | train {len(train_ids):,} chars | val {len(val_ids):,} chars")
print("round trip ok:", decode(encode("To be, or not")) == "To be, or not")


def make_streams(token_ids, batch_size):
    """Reshape a 1-D token tensor into (batch_size, n) contiguous streams, dropping the remainder."""
    n = token_ids.numel() // batch_size
    return token_ids[:n * batch_size].view(batch_size, n)


BATCH_LM = 32 if FAST_RUN else 64
BPTT = 64 if FAST_RUN else 128            # truncation length: steps unrolled per backward pass
train_streams = make_streams(train_ids, BATCH_LM).to(device)
val_streams = make_streams(val_ids, 8)
if FAST_RUN:
    val_streams = val_streams[:, :4000]   # evaluate on a slice to keep the fast run short
val_streams = val_streams.to(device)
print("train streams:", tuple(train_streams.shape), "| val streams:", tuple(val_streams.shape))

x_chunk = train_streams[0, :BPTT]
y_chunk = train_streams[0, 1:BPTT + 1]
print("input  chunk of stream 0:", repr(decode(x_chunk[:60])))
print("target chunk of stream 0:", repr(decode(y_chunk[:60])))

### 6.4 The model, the evaluation metric and sampling

The model is an embedding layer, one or two LSTM layers and a linear head that produces `VOCAB_SIZE` logits at every time step. Its `forward` accepts and returns the LSTM state so that the training loop can carry the state from one chunk to the next.

The standard evaluation metric for language models is **perplexity**, the exponential of the average per-token cross-entropy: $\text{PPL} = \exp\!\big(-\tfrac{1}{N}\sum_t \log p(x_t \mid x_{<t})\big)$. It is interpretable as an *effective branching factor*: a model with perplexity 4 is, on average, as uncertain about the next character as if it were choosing uniformly among four. A uniform model over 65 characters has perplexity 65; good character-level models on this corpus reach around 3–4, and word-level models on large corpora are measured in the tens (their tokens carry far more information each, so the numbers are not comparable across tokenisations).

To **sample** we feed a prompt, take the logits at the last position, divide them by a **temperature** $\tau$ and draw from the softmax, $p_i \propto \exp(z_i / \tau)$. With $\tau \to 0$ this becomes greedy decoding (always the most likely character); with $\tau = 1$ we sample from the model's own distribution; with $\tau > 1$ the distribution flattens toward uniform. Each drawn character is fed back in as the next input, one step at a time, carrying the state, so generation costs one cell step per character.

In [ ]:
class CharLSTM(nn.Module):
    """Embedding -> (multi-layer) LSTM -> linear head producing next-token logits at every step."""

    def __init__(self, vocab_size, emb_dim, hidden_size, num_layers=1, dropout=0.0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_size, num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, state=None):
        out, state = self.lstm(self.emb(x), state)   # out: (batch, time, hidden)
        return self.head(out), state                 # logits: (batch, time, vocab)


@torch.no_grad()
def evaluate_lm(model, streams, bptt):
    """Average next-token cross-entropy (nats/token) over the streams, carrying state across chunks."""
    was_training = model.training
    model.eval()
    B, n = streams.shape
    state, total, count = None, 0.0, 0
    for i in range(0, n - 1, bptt):
        x = streams[:, i:i + bptt]
        y = streams[:, i + 1:i + 1 + bptt]
        L = min(x.shape[1], y.shape[1])
        logits, state = model(x[:, :L], state)
        total += F.cross_entropy(logits.reshape(-1, logits.shape[-1]), y[:, :L].reshape(-1),
                                 reduction="sum").item()
        count += L * B
    if was_training:
        model.train()
    return total / count


@torch.no_grad()
def sample_ids(model, prompt_ids, length, temperature=1.0):
    """Autoregressively sample `length` token ids after a 1-D prompt tensor."""
    was_training = model.training
    model.eval()
    logits, state = model(prompt_ids.view(1, -1).to(device))
    generated = []
    for _ in range(length):
        probs = F.softmax(logits[0, -1] / temperature, dim=-1)
        nxt = torch.multinomial(probs, num_samples=1)         # (1,)
        generated.append(nxt.item())
        logits, state = model(nxt.view(1, 1), state)           # one step, carrying the state
    if was_training:
        model.train()
    return generated


def sample_text(model, prompt, length=300, temperature=1.0):
    return prompt + decode(sample_ids(model, encode(prompt), length, temperature))


probe_lm = CharLSTM(VOCAB_SIZE, emb_dim=16, hidden_size=32).to(device)
probe_loss = evaluate_lm(probe_lm, val_streams, BPTT)
print(f"untrained model: val loss {probe_loss:.3f} nats, perplexity {math.exp(probe_loss):.1f} "
      f"(uniform would be {VOCAB_SIZE})")
print("untrained sample:", repr(sample_text(probe_lm, text[:5], length=40)))

### 6.5 Training with truncated BPTT

The training loop below is the concrete form of Section 3.5. It walks through the chunks of the streams in order; for chunk $k$ it feeds the LSTM the state returned by chunk $k-1$, **detached** from the graph. The forward pass thus benefits from the entire history of each stream, while `backward()` only unrolls the `BPTT` steps of the current chunk. When the streams are exhausted we wrap around and reset the state to zeros, since the start of stream $b$ does not follow the end of stream $b$. Every step clips the gradient norm at 1. We log an exponential moving average of the training loss together with the validation loss and perplexity, and we store a text sample at a few checkpoints so that we can watch the model's writing improve.

In [ ]:
def train_lm(model, train_streams, val_streams, steps, bptt, lr, log_every, sample_steps, prompt):
    """Train a next-token model with truncated BPTT over contiguous streams."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    B, n = train_streams.shape
    n_chunks = max(1, (n - 1) // bptt)
    history = {"step": [], "train": [], "val": []}
    samples = {}
    state, running, t0 = None, None, time.time()
    model.train()
    for step in range(1, steps + 1):
        k = (step - 1) % n_chunks
        if k == 0:
            state = None                                   # new pass over the streams: reset state
        x = train_streams[:, k * bptt:(k + 1) * bptt]
        y = train_streams[:, k * bptt + 1:(k + 1) * bptt + 1]
        L = min(x.shape[1], y.shape[1])
        logits, state = model(x[:, :L], state)
        state = tuple(s.detach() for s in state)           # truncation: carry the state, cut the graph
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), y[:, :L].reshape(-1))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        running = loss.item() if running is None else 0.98 * running + 0.02 * loss.item()
        if step % log_every == 0 or step == steps:
            val_loss = evaluate_lm(model, val_streams, bptt)
            history["step"].append(step); history["train"].append(running); history["val"].append(val_loss)
            print(f"step {step:5d} | train loss {running:.3f} | val loss {val_loss:.3f} "
                  f"| val ppl {math.exp(val_loss):7.2f} | {time.time() - t0:6.1f}s")
        if step in sample_steps:
            samples[step] = sample_text(model, prompt, length=200, temperature=0.8)
    return history, samples

Under `FAST_RUN` the model is small (one layer of 128 units) and trains for 300 steps, which on a CPU takes well under a minute and is enough to see words appear. The full configuration (two layers of 384 units, dropout 0.2 between the layers, 2,500 steps of 64×128 characters) takes a few minutes on a GPU and reaches a validation perplexity around 4 (a cross-entropy of roughly 1.4 nats per character).

In [ ]:
torch.manual_seed(SEED)
if FAST_RUN:
    char_lm = CharLSTM(VOCAB_SIZE, emb_dim=32, hidden_size=128, num_layers=1).to(device)
    STEPS_LM, LR_LM, LOG_LM = 300, 3e-3, 50
else:
    char_lm = CharLSTM(VOCAB_SIZE, emb_dim=64, hidden_size=384, num_layers=2, dropout=0.2).to(device)
    STEPS_LM, LR_LM, LOG_LM = 2500, 2e-3, 250
set_forget_bias(char_lm.lstm, 1.0)
SAMPLE_STEPS = sorted({1, STEPS_LM // 4, STEPS_LM // 2, STEPS_LM})
PROMPT = "ROMEO:\n" if all(c in stoi for c in "ROMEO:\n") else text[:8]
print(f"model parameters: {sum(p.numel() for p in char_lm.parameters()):,} | "
      f"characters per step: {BATCH_LM * BPTT:,} | steps: {STEPS_LM}")
lm_history, lm_samples = train_lm(char_lm, train_streams, val_streams, STEPS_LM, BPTT, LR_LM,
                                  LOG_LM, SAMPLE_STEPS, PROMPT)

The validation loss should fall quickly during the first hundred steps (the model learns the character frequencies and the most common bigrams, which already halves the perplexity) and then more slowly as it picks up words and layout. The training and validation curves stay close on the full corpus: a million characters is a lot of data for a model of this size, so there is little overfitting; on the fallback text they diverge immediately, which is what memorising a few thousand characters looks like. Let us plot the curves and read the samples collected at the checkpoints.

In [ ]:
fig, ax = plt.subplots()
ax.plot(lm_history["step"], lm_history["train"], marker="o", label="train (moving average)")
ax.plot(lm_history["step"], lm_history["val"], marker="s", label="validation")
ax.set_xlabel("optimizer step"); ax.set_ylabel("cross-entropy (nats per character)")
ax.set_title(f"Character LSTM on {text_source}")
ax.legend(); plt.tight_layout(); plt.show()

val_ppl = math.exp(lm_history["val"][-1])
print(f"final validation perplexity: {val_ppl:.2f}  (uniform over {VOCAB_SIZE} characters: {VOCAB_SIZE})\n")
for step, sample in lm_samples.items():
    print(f"----- sample after step {step} (temperature 0.8) -----")
    print(sample)
    print()

Read the samples in order. After one step the model produces noise drawn from a nearly uniform distribution. A quarter of the way through it has learned the *texture* of the corpus: short lines, upper-case speaker names followed by a colon, spaces at reasonable intervals, vowels and consonants alternating. Halfway through, most "words" are pronounceable and many are real; by the end (on the full corpus) whole lines are grammatical fragments in a recognisable Elizabethan register, with the occasional invented word that looks like it *ought* to exist. The model has learned all this from nothing but the task of predicting the next character. What it has not learned is meaning: characters speak in turn but say nothing coherent, and a name introduced in one line is forgotten by the next. Larger models, longer contexts and, above all, the attention mechanism of Notebook 09 · Attention and Transformers improve this dramatically.

### 6.6 The effect of temperature

The next cell samples from the trained model at four temperatures, re-seeding before each sample so that the *only* difference is $\tau$.

In [ ]:
for temp in [0.3, 0.7, 1.0, 1.5]:
    torch.manual_seed(SEED)
    print(f"===== temperature {temp} =====")
    print(sample_text(char_lm, PROMPT, length=250, temperature=temp))
    print()

Low temperature makes the model conservative: it repeats the most frequent words and constructions, produces text that is locally very "correct" but bland and prone to loops (with $\tau \to 0$ the sampler becomes deterministic and typically falls into a cycle such as repeating the same line). Temperature 1 gives samples faithful to the model's own uncertainty, more varied and with more mistakes. High temperature flattens the distribution and the text degenerates into unusual characters and misspellings, because low-probability characters, which are low-probability for a good reason, are drawn far too often. Temperature is the simplest of a family of decoding controls (top-$k$, nucleus sampling, repetition penalties) that Notebook 09 · Attention and Transformers covers in detail.

> **Key idea.** A language model is a next-token classifier trained with cross-entropy over a huge number of positions. Truncated BPTT lets a recurrent model train on an unbounded stream with bounded memory, and sampling with temperature turns the classifier into a generator.

## 7. Time-series forecasting

### 7.1 Forecasting as supervised learning

Forecasting asks: given the recent past of a signal, what comes next? To attack it with the tools of this course we turn it into a supervised problem by **windowing**: every input is a window of the previous $W$ values, $(y_{t-W}, \dots, y_{t-1})$, and the target is $y_t$ (one-step-ahead forecasting). Sliding the window along the series produces thousands of training examples from a single long sequence. Any model that maps a window to a number will do, and our first comparison will pit an LSTM against an MLP on the flattened window, a baseline that is often surprisingly hard to beat, and against the *naive forecast* "tomorrow equals today", which every forecasting paper should report and many do not.

We generate a synthetic signal so that we know its structure: the sum of two sinusoids with different periods (50 and 13 steps, so the combined pattern only repeats every 650 steps), a slow linear trend, and Gaussian noise. The series is split **in time**, not at random: the first 70% for training, the next 15% for validation, the last 15% for test. Shuffling before splitting would put windows from the test period into the training set, overlapping almost entirely with their neighbours, and would report an accuracy that no real deployment could achieve.

In [ ]:
rng = np.random.default_rng(SEED)
N_SERIES = 3000
tt = np.arange(N_SERIES)
series = (np.sin(2 * np.pi * tt / 50) + 0.5 * np.sin(2 * np.pi * tt / 13 + 0.3)
          + 0.0002 * tt + 0.1 * rng.standard_normal(N_SERIES)).astype(np.float32)

N_TRAIN_TS, N_VAL_TS = 2100, 450
train_raw = series[:N_TRAIN_TS]
val_raw = series[N_TRAIN_TS:N_TRAIN_TS + N_VAL_TS]
test_raw = series[N_TRAIN_TS + N_VAL_TS:]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(tt, series, linewidth=0.8)
ax.axvspan(0, N_TRAIN_TS, alpha=0.08, color="tab:blue", label="train")
ax.axvspan(N_TRAIN_TS, N_TRAIN_TS + N_VAL_TS, alpha=0.15, color="tab:orange", label="validation")
ax.axvspan(N_TRAIN_TS + N_VAL_TS, N_SERIES, alpha=0.15, color="tab:green", label="test")
ax.set_xlabel("time step"); ax.set_ylabel("value"); ax.set_title("Synthetic multi-frequency signal with trend")
ax.legend(loc="upper left"); plt.tight_layout(); plt.show()
print(f"train {len(train_raw)} | val {len(val_raw)} | test {len(test_raw)} points")

### 7.2 Scaling, and the pitfall of fitting it on everything

Neural networks train best on inputs of roughly unit scale, so we standardise the series. The scaler's mean and standard deviation must be computed **on the training portion only** and then applied unchanged to validation and test. Fitting it on the whole series is a subtle form of leakage: the test-period mean would shift the training inputs and tell the model, before it has seen a single test point, where the series is heading. The effect is small on a stationary signal and large on one with a trend, and it is one of the most common mistakes in published forecasting comparisons. The cell prints both versions of the statistics so that you can see they differ.

Windowing then produces `(n_windows, W, 1)` inputs. For validation and test we let the windows start `W` steps *before* the split so that every validation and test point is a target exactly once, while the model never sees a target from a later period than its inputs.

In [ ]:
mu, sigma = float(train_raw.mean()), float(train_raw.std())
mu_all, sigma_all = float(series.mean()), float(series.std())
print(f"scaler fitted on train only: mean {mu:+.3f}, std {sigma:.3f}")
print(f"scaler fitted on all data  : mean {mu_all:+.3f}, std {sigma_all:.3f}   <- leaks the trend")
scaled = (series - mu) / sigma                      # same transform applied everywhere

WINDOW = 48

def make_windows(x, window):
    """Sliding windows of length `window`; the target is the value right after each window."""
    X = np.stack([x[i:i + window] for i in range(len(x) - window)])
    y = x[window:]
    return (torch.tensor(X, dtype=torch.float32)[..., None],   # (n, window, 1)
            torch.tensor(y, dtype=torch.float32)[:, None])      # (n, 1)

X_tr, y_tr = make_windows(scaled[:N_TRAIN_TS], WINDOW)
X_va, y_va = make_windows(scaled[N_TRAIN_TS - WINDOW:N_TRAIN_TS + N_VAL_TS], WINDOW)
X_te, y_te = make_windows(scaled[N_TRAIN_TS + N_VAL_TS - WINDOW:], WINDOW)
print("windows: train", tuple(X_tr.shape), "| val", tuple(X_va.shape), "| test", tuple(X_te.shape))

pin = (device.type == "cuda")
train_loader_ts = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True, num_workers=0, pin_memory=pin)
val_loader_ts = DataLoader(TensorDataset(X_va, y_va), batch_size=256, shuffle=False, num_workers=0, pin_memory=pin)
test_loader_ts = DataLoader(TensorDataset(X_te, y_te), batch_size=256, shuffle=False, num_workers=0, pin_memory=pin)

### 7.3 LSTM versus MLP versus the naive forecast

The MLP flattens the window into a 48-vector and applies two linear layers with a ReLU. The LSTM reads the window one value at a time and predicts from its final state. Both are trained with Adam on the mean squared error in scaled units; we report the **mean absolute error in original units** on the test set, which is what a user of the forecast cares about, next to the naive forecast's error.

In [ ]:
class MLPForecaster(nn.Module):
    def __init__(self, window, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(window, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, x):                 # x: (batch, window, 1)
        return self.net(x)


class LSTMForecaster(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):                 # x: (batch, window, 1)
        out, _ = self.lstm(x)
        return self.head(out[:, -1])


@torch.no_grad()
def predict(model, loader):
    """Return (predictions, targets) as flat NumPy arrays in scaled units."""
    model.eval()
    preds, targets = [], []
    for xb, yb in loader:
        preds.append(model(xb.to(device)).cpu())
        targets.append(yb)
    p = torch.cat(preds).squeeze(-1)
    y = torch.cat(targets).squeeze(-1)
    return p.cpu().numpy(), y.cpu().numpy()


def train_forecaster(model, train_loader, val_loader, epochs, lr=2e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train": [], "val": []}
    for epoch in range(1, epochs + 1):
        model.train()
        total, count, t0 = 0.0, 0, time.time()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = F.mse_loss(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * len(xb); count += len(xb)
        p, y = predict(model, val_loader)
        val_mse = float(np.mean((p - y) ** 2))
        history["train"].append(total / count); history["val"].append(val_mse)
        if epoch == 1 or epoch % max(1, epochs // 5) == 0 or epoch == epochs:
            print(f"  epoch {epoch:3d} | train mse {total / count:.4f} | val mse {val_mse:.4f} | {time.time() - t0:.1f}s")
    return history


EPOCHS_TS = 8 if FAST_RUN else 30
torch.manual_seed(SEED)
mlp_fc = MLPForecaster(WINDOW).to(device)
print("MLP forecaster:")
hist_mlp = train_forecaster(mlp_fc, train_loader_ts, val_loader_ts, EPOCHS_TS, lr=2e-3)
torch.manual_seed(SEED)
lstm_fc = LSTMForecaster().to(device)
print("LSTM forecaster:")
hist_lstm = train_forecaster(lstm_fc, train_loader_ts, val_loader_ts, EPOCHS_TS, lr=5e-3)

p_mlp, y_test_scaled = predict(mlp_fc, test_loader_ts)
p_lstm, _ = predict(lstm_fc, test_loader_ts)
p_naive = X_te[:, -1, 0].cpu().numpy()    # "next value = last value"
mae = lambda p: float(np.mean(np.abs(p - y_test_scaled)) * sigma)
print(f"\nTest MAE in original units: naive {mae(p_naive):.4f} | MLP {mae(p_mlp):.4f} | LSTM {mae(p_lstm):.4f}"
      f"  (noise std = 0.100)")

Both learned models should beat the naive forecast clearly; the signal is smooth enough that "same as yesterday" is a poor predictor, but on a random-walk-like series (stock prices, for instance) the naive forecast is nearly unbeatable and every reported gain must be measured against it. The floor for any one-step forecaster is the noise standard deviation of 0.1: no model can predict the noise term that is added fresh at every step, so an MAE near $0.1 \cdot \sqrt{2/\pi} \approx 0.08$ is essentially perfect. Which of the two gets closer depends on the budget: the MLP has direct access to all 48 lags and fits within a couple of epochs, whereas the LSTM must learn to summarise the window through its state, needs a higher learning rate and more epochs to get there, and benefits most from the full run (if its MAE is still near the naive baseline under `FAST_RUN`, look at the validation MSE printed per epoch: it is still falling, i.e. the model is undertrained rather than incapable). On this simple, stationary-apart-from-trend signal the two end up close; the LSTM's advantages appear with longer windows, variable-length histories and multivariate inputs, where the MLP's parameter count grows with the window while the LSTM's does not. Let us look at the one-step predictions on a stretch of the test period.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
show = slice(0, 200)
t_test = np.arange(N_TRAIN_TS + N_VAL_TS, N_SERIES)[show]
ax.plot(t_test, y_test_scaled[show] * sigma + mu, "k", label="truth", linewidth=1.5)
ax.plot(t_test, p_mlp[show] * sigma + mu, label="MLP (one-step)", alpha=0.8)
ax.plot(t_test, p_lstm[show] * sigma + mu, label="LSTM (one-step)", alpha=0.8)
ax.set_xlabel("time step"); ax.set_ylabel("value"); ax.set_title("One-step-ahead forecasts on the test period")
ax.legend(); plt.tight_layout(); plt.show()

### 7.4 Multi-step forecasting

One-step-ahead accuracy is flattering: the model always gets the *true* recent past as input. A real forecast must look further ahead, and the simplest way to do so with a one-step model is **autoregressive rollout**: predict $\hat y_{t}$, append it to the window as if it were observed, predict $\hat y_{t+1}$ from the window that now contains one prediction, and so on. Errors compound, because each prediction is conditioned on earlier predictions rather than on data. The alternative is **direct multi-step** forecasting, training a model whose output is the whole vector of the next $H$ values, which avoids compounding at the price of $H$ times more targets to learn; a seq2seq model (Section 9) is the sequence-model version of this idea.

We roll both models out for 100 steps from the last window of the validation period into the test period and compare with the truth.

In [ ]:
@torch.no_grad()
def rollout(model, context, horizon):
    """Autoregressive forecast: feed each prediction back as the newest input. Returns scaled values."""
    model.eval()
    window = torch.tensor(context, dtype=torch.float32, device=device)   # (WINDOW,)
    preds = []
    for _ in range(horizon):
        nxt = model(window[None, :, None])[0, 0]                          # (1, WINDOW, 1) -> scalar
        preds.append(nxt.item())
        window = torch.cat([window[1:], nxt.view(1)])
    return np.array(preds)


HORIZON = 100
context = scaled[N_TRAIN_TS + N_VAL_TS - WINDOW:N_TRAIN_TS + N_VAL_TS]
truth = test_raw[:HORIZON]
roll_mlp = rollout(mlp_fc, context, HORIZON) * sigma + mu
roll_lstm = rollout(lstm_fc, context, HORIZON) * sigma + mu
t_roll = np.arange(N_TRAIN_TS + N_VAL_TS, N_TRAIN_TS + N_VAL_TS + HORIZON)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(t_roll, truth, "k", linewidth=1.5, label="truth")
ax.plot(t_roll, roll_mlp, label=f"MLP rollout (MAE {np.mean(np.abs(roll_mlp - truth)):.3f})")
ax.plot(t_roll, roll_lstm, label=f"LSTM rollout (MAE {np.mean(np.abs(roll_lstm - truth)):.3f})")
ax.set_xlabel("time step"); ax.set_ylabel("value"); ax.set_title(f"{HORIZON}-step autoregressive forecasts")
ax.legend(); plt.tight_layout(); plt.show()

for h in [1, 10, 25, 50, 100]:
    print(f"horizon {h:3d}: |error| MLP {abs(roll_mlp[h-1] - truth[h-1]):.3f} | LSTM {abs(roll_lstm[h-1] - truth[h-1]):.3f}")

The better one-step model tracks the truth for a period or two of the slow sinusoid and then gradually drifts: phase errors accumulate, the fast oscillation is the first to lose alignment, and the amplitude often decays because each model, trained to minimise squared error, hedges towards the mean when its inputs become unfamiliar. The weaker one-step model (under `FAST_RUN` usually the undertrained LSTM) drifts much sooner, because a rollout amplifies exactly the one-step errors that looked harmless in the previous plot. The trend is the other failure mode to watch: the models have only seen the level of the training period, and a forecast far into a trending future extrapolates beyond anything in the training inputs. Detrending or differencing the series before modelling, and adding the trend back afterwards, is the standard remedy, and it is the single most effective preprocessing step in classical forecasting.

> **Common pitfall.** Always evaluate forecasting models in the way they will be used. A one-step-ahead MSE says nothing about a 30-day forecast, a random train/test split leaks the future, and a scaler fitted on all data leaks the level. The three mistakes together can make a useless model look excellent.

## 8. Sequence classification with variable-length inputs

### 8.1 Padding, masking and packing

Real sequences come in different lengths, but a tensor must be rectangular. The universal solution is **padding**: every sequence in a mini-batch is extended with a special `PAD` token (index 0 by convention) up to the length of the longest one, and a vector of true **lengths** travels with the batch. Padding creates two problems that must be solved explicitly, because PyTorch will happily process pad tokens as if they were data.

The first is in the model. If we read a many-to-one prediction from the final hidden state, the "final" step of a padded sequence is a pad, and the state has been updated several times on meaningless input since the true end. Worse, for a *bidirectional* model the backward direction *starts* on the pads. The fix is **packing**: `pack_padded_sequence` converts the padded tensor plus lengths into a `PackedSequence`, a representation in which the recurrent layer knows where each sequence ends, stops updating that sequence's state there, and returns `h_n` taken at the true last step (and, for the backward direction, run from the true last step to the first). `pad_packed_sequence` converts the outputs back to a padded tensor with zeros beyond each length.

The second is in any pooling or loss over time. A mean over time must divide by the true length and exclude pads, which is done with a boolean **mask** of shape `(batch, time)`; a per-step loss must ignore pad targets (`ignore_index` in `F.cross_entropy`). Attention masks in Notebook 09 · Attention and Transformers are the same idea.

### 8.2 A synthetic classification task

We build a task with a hidden "sentiment lexicon": each of 20 token types carries a fixed score in $\{-1, 0, +1\}$ that the model does not know; a sequence of 4–30 tokens is labelled positive if the sum of its scores is positive. The model must learn the lexicon *and* to aggregate it over a variable length. Tokens are shifted by one so that index 0 is free for padding.

In [ ]:
VOCAB_CLS, PAD_ID = 20, 0
torch.manual_seed(SEED)
token_score = torch.tensor([-1] * 7 + [0] * 6 + [1] * 7)[torch.randperm(VOCAB_CLS)]   # balanced lexicon
print("hidden token scores:", token_score.tolist())

def make_varlen_dataset(n, min_len=4, max_len=30):
    """Variable-length token sequences (ids 1..VOCAB_CLS, 0 reserved for PAD) with binary labels."""
    seqs, labels = [], []
    while len(seqs) < n:
        L = random.randint(min_len, max_len)
        toks = torch.randint(0, VOCAB_CLS, (L,))
        s = int(token_score[toks].sum())
        if s == 0:                      # discard ties so that every label is unambiguous
            continue
        seqs.append(toks + 1)
        labels.append(int(s > 0))
    return seqs, labels


def collate_varlen(batch):
    """Pad a list of (sequence, label) pairs into (padded ids, lengths, labels)."""
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    padded = pad_sequence(list(seqs), batch_first=True, padding_value=PAD_ID)
    return padded, lengths, torch.tensor(labels)


random.seed(SEED); torch.manual_seed(SEED)
train_seqs, train_labels = make_varlen_dataset(4000)
test_seqs, test_labels = make_varlen_dataset(1000)
pin = (device.type == "cuda")
train_loader_cls = DataLoader(list(zip(train_seqs, train_labels)), batch_size=64, shuffle=True,
                              collate_fn=collate_varlen, num_workers=0, pin_memory=pin)
test_loader_cls = DataLoader(list(zip(test_seqs, test_labels)), batch_size=256, shuffle=False,
                             collate_fn=collate_varlen, num_workers=0, pin_memory=pin)

x_pad, lengths, y_cls = next(iter(test_loader_cls))
print("padded batch:", tuple(x_pad.shape), "| lengths of the first 6:", lengths[:6].tolist(),
      "| positive fraction: %.2f" % np.mean(train_labels))
print("first sequence (padded):", x_pad[0].tolist())

### 8.3 Why packing matters: a direct check

Before training anything, let us verify the claim that padding corrupts the final state and packing repairs it. We take one sequence, run an (untrained) LSTM over it alone, then over the same sequence followed by ten pads, and finally over the padded version *packed* with its true length. Note that the pad embedding is the zero vector (`padding_idx=0`), yet the recurrent update on a zero input still changes the state through the biases and the recurrent weights.

In [ ]:
torch.manual_seed(SEED)
demo_emb = nn.Embedding(VOCAB_CLS + 1, 4, padding_idx=PAD_ID)
demo_lstm = nn.LSTM(4, 6, batch_first=True)
seq = train_seqs[0]
with torch.no_grad():
    alone = demo_emb(seq[None])                                                  # (1, L, 4)
    padded = demo_emb(torch.cat([seq, torch.zeros(10, dtype=torch.long)])[None])   # (1, L+10, 4)
    _, (h_alone, _) = demo_lstm(alone)
    _, (h_padded, _) = demo_lstm(padded)
    packed = pack_padded_sequence(padded, torch.tensor([len(seq)]), batch_first=True, enforce_sorted=False)
    _, (h_packed, _) = demo_lstm(packed)
print(f"sequence length {len(seq)}, padded to {len(seq) + 10}")
print(f"|h(alone) - h(padded)| max = {(h_alone - h_padded).abs().max().item():.4f}   <- padding changed the state")
print(f"|h(alone) - h(packed)| max = {(h_alone - h_packed).abs().max().item():.1e}   <- packing recovers it")

### 8.4 A bidirectional LSTM classifier, with and without packing

The classifier embeds the tokens, runs a bidirectional LSTM, concatenates the final forward state (`h_n[-2]`, taken at each sequence's true end when packed) with the final backward state (`h_n[-1]`, taken at position 0), and applies a linear layer. A flag switches packing on or off so that we can measure the difference on identical data and seeds. The `lengths` tensor is passed to `pack_padded_sequence` on the CPU, as the function requires, and `enforce_sorted=False` lets us keep the batch in its original order.

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim=16, hidden_size=32, use_packing=True):
        super().__init__()
        self.use_packing = use_packing
        self.emb = nn.Embedding(vocab_size + 1, emb_dim, padding_idx=PAD_ID)
        self.lstm = nn.LSTM(emb_dim, hidden_size, batch_first=True, bidirectional=True)
        self.head = nn.Linear(2 * hidden_size, 2)

    def forward(self, x, lengths):
        e = self.emb(x)
        if self.use_packing:
            packed = pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
            _, (h_n, _) = self.lstm(packed)
        else:
            _, (h_n, _) = self.lstm(e)
        feat = torch.cat([h_n[-2], h_n[-1]], dim=-1)   # forward and backward final states of the last layer
        return self.head(feat)


def run_epoch_cls(model, loader, opt=None):
    """One pass over `loader`; trains if an optimizer is given. Returns (mean loss, accuracy)."""
    training = opt is not None
    model.train(training)
    total, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, lengths, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x, lengths)
            loss = F.cross_entropy(logits, y)
            if training:
                opt.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            total += loss.item() * len(y)
            correct += (logits.argmax(dim=-1) == y).sum().item()
            count += len(y)
    return total / count, correct / count


EPOCHS_CLS = 2 if FAST_RUN else 8
cls_models = {}
for use_packing in [True, False]:
    torch.manual_seed(SEED)
    model = BiLSTMClassifier(VOCAB_CLS, use_packing=use_packing).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    name = "packed" if use_packing else "padded (no packing)"
    print(f"{name}:")
    for epoch in range(1, EPOCHS_CLS + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch_cls(model, train_loader_cls, opt)
        te_loss, te_acc = run_epoch_cls(model, test_loader_cls)
        print(f"  epoch {epoch} | train loss {tr_loss:.3f} acc {tr_acc:.3f} | test loss {te_loss:.3f} acc {te_acc:.3f} | {time.time() - t0:.1f}s")
    cls_models[name] = model

The packed model has an easier job: it reads exactly the sequence and nothing else, and it typically learns faster (compare the accuracies after the first epoch). On this easy task both models usually end near 100% after two epochs, because the LSTM is able to discover that pad inputs should be ignored (the forget gate can learn to stay open on zero input) and the backward direction can learn to treat leading pads as a no-op; but that is capacity and training time spent on undoing an artefact of batching. On harder tasks, longer sequences with more padding, and smaller datasets the gap grows, and for models that read outputs at the last time step *without* `h_n` (for example `out[:, -1]`), the unpacked version is simply wrong, because that position is a pad for most sequences.

### 8.5 Masked pooling over time

Instead of the final states one often pools the outputs over time, which uses information from every position and is more robust. With packed input, `pad_packed_sequence` returns the per-step outputs with exact zeros beyond each sequence's length, and a mask built from the lengths implements a mean over the valid positions only.

In [ ]:
x_pad, lengths, y_cls = next(iter(test_loader_cls))
model = cls_models["packed"]
with torch.no_grad():
    e = model.emb(x_pad.to(device))
    packed = pack_padded_sequence(e, lengths, batch_first=True, enforce_sorted=False)
    out_packed, _ = model.lstm(packed)
    out, out_lengths = pad_packed_sequence(out_packed, batch_first=True)     # (B, L_max, 2H)
lengths_dev = lengths.to(out.device)
mask = torch.arange(out.shape[1], device=out.device)[None, :] < lengths_dev[:, None]   # (B, L_max) bool
print("outputs:", tuple(out.shape), "| mask:", tuple(mask.shape), "| valid positions:", int(mask.sum()), "of", mask.numel())
print("outputs beyond each sequence's length are exactly zero:", bool((out[~mask] == 0).all()))
pooled = (out * mask[..., None]).sum(dim=1) / lengths_dev[:, None]                # masked mean over time
print("masked mean pooling ->", tuple(pooled.shape))

> **Key idea.** Padding makes batches rectangular; lengths, masks and `PackedSequence` make the computation *ignore* the padding. Every sequence model in this course, recurrent or Transformer, needs one of these mechanisms, and forgetting it is a silent bug: the code runs and the metrics are merely worse than they should be.

## 9. Seq2seq and the bottleneck

### 9.1 Encoder–decoder models

When the output is itself a sequence of a different length (translation, summarisation, speech-to-text), the model needs to read the whole input before it starts writing. The **encoder–decoder** or **sequence-to-sequence** architecture (Sutskever, Vinyals and Le, 2014; Cho et al., 2014) uses two recurrent networks: an *encoder* reads the source and produces a final state, and a *decoder*, initialised with that state, generates the target one token at a time as a language model conditioned on the source. The decoder starts from a start-of-sequence token `<sos>` and stops when it emits `<eos>`.

Training uses **teacher forcing**: at every decoder step the input is the *true* previous target token, not the token the decoder generated, so the whole target sequence can be processed in one parallel pass and the loss is the ordinary per-position cross-entropy. At inference we have no true tokens and must feed back the model's own predictions (greedy decoding here; beam search is discussed in Notebook 09 · Attention and Transformers). The mismatch between the two regimes, training on gold prefixes but generating on its own imperfect ones, is called *exposure bias*; scheduled sampling and, later, sequence-level objectives were proposed to reduce it.

### 9.2 A toy task: reversing digit sequences

The task is to output the input sequence of digits in reverse order. It looks trivial but is a demanding memory test for the encoder: the *entire* sequence must be stored in one fixed-size vector before the first output can be produced, and the last input digit must be recalled first. We measure exact-match accuracy as a function of the sequence length, on lengths seen in training and on a few longer ones.

In [ ]:
PAD, SOS, EOS = 0, 1, 2
N_DIGITS = 10
V_S2S = 3 + N_DIGITS          # special tokens + digits (digit d has id d + 3)

def make_reverse_batch(batch_size, L):
    """Source digit sequences of length L; decoder input = <sos> + reversed; decoder target = reversed + <eos>."""
    src = torch.randint(3, V_S2S, (batch_size, L))
    tgt = torch.flip(src, dims=[1])
    dec_in = torch.cat([torch.full((batch_size, 1), SOS), tgt], dim=1)    # (B, L+1)
    dec_out = torch.cat([tgt, torch.full((batch_size, 1), EOS)], dim=1)   # (B, L+1)
    return src, dec_in, dec_out


def show_tokens(ids):
    names = {PAD: "<pad>", SOS: "<sos>", EOS: "<eos>"}
    return " ".join(names.get(int(i), str(int(i) - 3)) for i in ids)


src, dec_in, dec_out = make_reverse_batch(1, 6)
print("source        :", show_tokens(src[0]))
print("decoder input :", show_tokens(dec_in[0]))
print("decoder target:", show_tokens(dec_out[0]))

The encoder returns both its per-step outputs (unused here, needed for attention in Exercise 5) and its final state; the decoder is a GRU language model that starts from that state. `greedy_decode` implements inference: feed `<sos>`, take the argmax, feed it back, repeat.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.gru = nn.GRU(emb_dim, hidden_size, batch_first=True)

    def forward(self, src):
        out, h = self.gru(self.emb(src))       # out: (B, L, H) all states; h: (1, B, H) final state
        return out, h


class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.gru = nn.GRU(emb_dim, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, vocab_size)

    def forward(self, dec_in, h):
        out, h = self.gru(self.emb(dec_in), h)
        return self.out(out), h                 # logits: (B, T, vocab)


class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, emb_dim=32, hidden_size=64):
        super().__init__()
        self.encoder = Encoder(vocab_size, emb_dim, hidden_size)
        self.decoder = Decoder(vocab_size, emb_dim, hidden_size)

    def forward(self, src, dec_in):             # teacher forcing: the whole target in one pass
        _, h = self.encoder(src)
        logits, _ = self.decoder(dec_in, h)
        return logits

    @torch.no_grad()
    def greedy_decode(self, src, max_len):
        was_training = self.training
        self.eval()
        _, h = self.encoder(src)
        y = torch.full((src.shape[0], 1), SOS, dtype=torch.long, device=src.device)
        outputs = []
        for _ in range(max_len):
            logits, h = self.decoder(y, h)
            y = logits[:, -1].argmax(dim=-1, keepdim=True)
            outputs.append(y)
        self.train(was_training)                # restore whatever mode the caller had set
        return torch.cat(outputs, dim=1)


def train_seq2seq(model, steps, max_len, lr=3e-3, batch_size=64, log_every=100):
    """Train on random reversal batches whose length is drawn uniformly from 3..max_len."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    losses, t0 = [], time.time()
    for step in range(1, steps + 1):
        L = random.randint(3, max_len)
        src, dec_in, dec_out = (t.to(device) for t in make_reverse_batch(batch_size, L))
        logits = model(src, dec_in)
        loss = F.cross_entropy(logits.reshape(-1, V_S2S), dec_out.reshape(-1))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        losses.append(loss.item())
        if step % log_every == 0 or step == steps:
            print(f"  step {step:5d} | loss {np.mean(losses[-log_every:]):.3f} | {time.time() - t0:5.1f}s")
    return losses


@torch.no_grad()
def accuracy_by_length(model, lengths, n=500):
    """Exact-match accuracy of greedy decoding for each sequence length."""
    accs = []
    for L in lengths:
        src, _, _ = make_reverse_batch(n, L)
        src = src.to(device)
        pred = model.greedy_decode(src, L)
        accs.append((pred == torch.flip(src, dims=[1])).all(dim=1).float().mean().item())
    return accs


L_MAX_S2S = 10 if FAST_RUN else 16
STEPS_S2S = 400 if FAST_RUN else 3000
torch.manual_seed(SEED); random.seed(SEED)
s2s = Seq2Seq(V_S2S, hidden_size=64 if FAST_RUN else 128).to(device)
print(f"training plain seq2seq on lengths 3..{L_MAX_S2S}")
s2s_losses = train_seq2seq(s2s, STEPS_S2S, L_MAX_S2S)

### 9.3 Accuracy versus length: the bottleneck

We evaluate exact-match accuracy for every length the model trained on and for four longer lengths it never saw.

In [ ]:
EVAL_LENGTHS = list(range(3, L_MAX_S2S + 5))
acc_s2s = accuracy_by_length(s2s, EVAL_LENGTHS)
for L, a in zip(EVAL_LENGTHS, acc_s2s):
    print(f"length {L:2d}: exact match {a:.3f}" + ("   (longer than any training sequence)" if L > L_MAX_S2S else ""))

src, _, _ = make_reverse_batch(1, L_MAX_S2S)
pred = s2s.greedy_decode(src.to(device), L_MAX_S2S)
print("\nsource :", show_tokens(src[0]))
print("target :", show_tokens(torch.flip(src, dims=[1])[0]))
print("greedy :", show_tokens(pred[0]))

fig, ax = plt.subplots()
ax.plot(EVAL_LENGTHS, acc_s2s, marker="o", label="plain seq2seq (GRU)")
ax.axvline(L_MAX_S2S + 0.5, color="gray", linestyle=":", label="max training length")
ax.set_xlabel("sequence length"); ax.set_ylabel("exact-match accuracy"); ax.set_ylim(0, 1.05)
ax.set_title("Reversal task: accuracy vs. length"); ax.legend(); plt.tight_layout(); plt.show()

Short sequences are reversed perfectly and accuracy falls with length, first gently, then steeply beyond the training range. The cause is the **bottleneck**: the whole source, however long, must be squeezed into one hidden vector of fixed size, and the decoder must then unpack it in the right order from that vector alone. Information about early input digits, which are needed *last*, has to survive the encoding of everything that follows and then the generation of everything that precedes them. Larger hidden states and more training postpone the problem; they do not remove it. Machine translation systems of 2014 had exactly this signature: excellent on short sentences, degrading on long ones.

The remedy, proposed by Bahdanau, Cho and Bengio (2015), is to let the decoder *look back* at all of the encoder's per-step outputs at every generation step, and to learn where to look: at each step it computes a similarity score between its current state and every encoder state, normalises the scores with a softmax into weights, and reads a weighted average of the encoder states as extra input. There is no longer a single vector that must hold everything, and for the reversal task the learned weights become a clean anti-diagonal (output step $t$ attends to input position $L + 1 - t$, counting both from one). This mechanism is **attention**. Exercise 5 implements it on this task; Notebook 09 · Attention and Transformers then makes attention the *only* mechanism, removing the recurrence altogether.

> **Key idea.** A fixed-size state is a bottleneck between a variable-length input and a variable-length output. Attention replaces "compress everything into one vector" with "keep everything and learn what to read", and that idea is the foundation of the rest of Part IV.

## Exercises

**Exercise 1 (warm-up).** Implement a GRU cell from scratch following the equations of Section 4.6 and PyTorch's parametrisation (gates stacked in the order reset, update, candidate; `weight_ih` of shape `(3H, D)`, `weight_hh` of shape `(3H, H)`, two bias vectors). Copy the weights of an `nn.GRUCell` into it and verify that the hidden states agree over a sequence of ten steps.

**Solution to Exercise 1**

The only subtlety is where the reset gate acts: PyTorch multiplies $r_t$ with the *recurrent* part of the candidate, $W_{hn} h_{t-1} + b_{hn}$, including its bias, before the $\tanh$. Getting that detail wrong gives a different (also valid) GRU variant whose outputs will not match.

In [ ]:
class GRUCellScratch(nn.Module):
    """One GRU step with PyTorch's parametrisation (gates stacked as r, z, n)."""

    def __init__(self, input_size, hidden_size):
        super().__init__()
        k = 1.0 / math.sqrt(hidden_size)
        self.W_ih = nn.Parameter(torch.empty(3 * hidden_size, input_size).uniform_(-k, k))
        self.W_hh = nn.Parameter(torch.empty(3 * hidden_size, hidden_size).uniform_(-k, k))
        self.b_ih = nn.Parameter(torch.zeros(3 * hidden_size))
        self.b_hh = nn.Parameter(torch.zeros(3 * hidden_size))

    def forward(self, x_t, h_prev):
        gi = x_t @ self.W_ih.T + self.b_ih          # input contributions to r, z, n
        gh = h_prev @ self.W_hh.T + self.b_hh       # recurrent contributions to r, z, n
        i_r, i_z, i_n = gi.chunk(3, dim=-1)
        h_r, h_z, h_n = gh.chunk(3, dim=-1)
        r = torch.sigmoid(i_r + h_r)
        z = torch.sigmoid(i_z + h_z)
        n = torch.tanh(i_n + r * h_n)               # reset gate applied inside the tanh, to the recurrent part
        return (1 - z) * n + z * h_prev


torch.manual_seed(SEED)
D, H, B, T = 6, 10, 4, 10
ref_gru = nn.GRUCell(D, H)
our_gru = GRUCellScratch(D, H)
with torch.no_grad():
    our_gru.W_ih.copy_(ref_gru.weight_ih); our_gru.W_hh.copy_(ref_gru.weight_hh)
    our_gru.b_ih.copy_(ref_gru.bias_ih);   our_gru.b_hh.copy_(ref_gru.bias_hh)
x = torch.randn(B, T, D)
h_a = h_b = torch.zeros(B, H)
max_diff = 0.0
for t in range(T):
    h_a = our_gru(x[:, t], h_a)
    h_b = ref_gru(x[:, t], h_b)
    max_diff = max(max_diff, (h_a - h_b).abs().max().item())
print(f"max |difference| over {T} steps: {max_diff:.2e}")
print("parameters:", sum(p.numel() for p in our_gru.parameters()), "= 3*(H*D + H*H + 2H) =", 3 * (H * D + H * H + 2 * H))

**Exercise 2.** Compare gradient clipping by norm with clipping by value. (a) Take a gradient with a large norm (reuse `probe` from Section 3.5), apply `clip_grad_norm_(…, 1.0)` and `clip_grad_value_(…, 0.05)` to copies of it, and measure the cosine similarity of each clipped gradient with the original: which one preserves the direction? (b) Train the vanilla RNN on the delayed-recall task at $T = 15$ with plain SGD with momentum (`optimizer="sgd"`, learning rate 0.3), where the step is a fixed multiple of the (momentum-averaged) gradient, using `clip="norm"`, `clip="value"` and `clip=None` (the `train_recall` function supports all of these), and compare the test accuracies. Then repeat the three runs with Adam and explain the difference.

**Solution to Exercise 2**

Clipping by norm rescales all coordinates by the same factor, so the direction is exactly preserved (cosine similarity 1). Clipping by value saturates the large coordinates and leaves small ones untouched, which rotates the gradient towards the diagonal directions; the cosine similarity drops noticeably. With SGD the consequences are visible in the accuracies: the unclipped run is derailed by the occasional exploding gradient (a single step of norm 20 at learning rate 0.3 throws the weights far from where they were), clipping by value survives but learns worse because the rotated updates point in the wrong direction, and clipping by norm trains reliably. With Adam the three runs end up close to one another, because Adam's per-coordinate normalisation by the running second moment already caps the step size; that is why clipping matters less with adaptive optimizers, and why it is nevertheless kept as insurance for the moment estimates (Section 3.5).

In [ ]:
def flat_grad(model):
    return torch.cat([p.grad.flatten() for p in model.parameters()])

gradients_of(probe)
g_orig = flat_grad(probe).clone()
nn.utils.clip_grad_norm_(probe.parameters(), 1.0)
g_norm = flat_grad(probe).clone()
gradients_of(probe)
nn.utils.clip_grad_value_(probe.parameters(), 0.05)
g_value = flat_grad(probe).clone()
cos = lambda a, b: F.cosine_similarity(a, b, dim=0).item()
print(f"original norm {g_orig.norm():.2f} | by-norm: norm {g_norm.norm():.2f}, cosine {cos(g_orig, g_norm):.4f} "
      f"| by-value: norm {g_value.norm():.2f}, cosine {cos(g_orig, g_value):.4f}")

STEPS_EX2 = 300 if FAST_RUN else 1500
for optimizer, lr in [("sgd", 0.3), ("adam", 5e-3)]:
    for clip_mode in ["norm", "value", None]:
        acc = train_recall("RNN", T=15, steps=STEPS_EX2, lr=lr, clip=clip_mode, optimizer=optimizer)
        print(f"vanilla RNN, T=15, {optimizer:4s} lr {lr:<5} clip={str(clip_mode):5s}: test accuracy {acc:.3f}")

**Exercise 3.** Build a *word-level* language model on the same corpus. Tokenise the lower-cased text into words and punctuation with a regular expression, keep the 1,999 most frequent types and map everything else to `<unk>` (a vocabulary of 2,000 ids), and train the `CharLSTM` class (it is agnostic to what the tokens mean) with `train_lm` on streams of word ids. Report the validation perplexity and generate 40 words. Why is the perplexity so much larger than the character model's, and why is that not a fair comparison?

**Solution to Exercise 3**

The model and the training loop need no change at all; only the tokenizer and the vocabulary differ. Word-level perplexities are in the tens or hundreds while character-level ones are in the single digits, because a word carries several characters' worth of information: the per-*character* cross-entropy of a word model, obtained by dividing its per-word cross-entropy by the average word length, is the comparable number. The price of word-level modelling is the `<unk>` token: everything outside the 2,000-word vocabulary is invisible to the model, which is why subword tokenisation (Notebook 10 · NLP with Pretrained Transformers (Hugging Face)) took over.

In [ ]:
words = re.findall(r"[a-z']+|[.,;:!?]", text.lower())
V_WORDS = 2000
counter = Counter(words)
vocab_w = ["<unk>"] + [w for w, _ in counter.most_common(V_WORDS - 1)]
w2i = {w: i for i, w in enumerate(vocab_w)}
word_ids = torch.tensor([w2i.get(w, 0) for w in words], dtype=torch.long)
print(f"{len(words):,} word tokens | {len(counter):,} distinct | vocabulary {len(vocab_w)} "
      f"| coverage {(word_ids != 0).float().mean().item():.1%} | mean word length {np.mean([len(w) for w in words]):.2f}")

n_w = int(0.9 * len(word_ids))
w_train = make_streams(word_ids[:n_w], 16).to(device)
w_val = make_streams(word_ids[n_w:], 8)
w_val = (w_val[:, :2000] if FAST_RUN else w_val).to(device)

torch.manual_seed(SEED)
word_lm = CharLSTM(len(vocab_w), emb_dim=64, hidden_size=128 if FAST_RUN else 256).to(device)
set_forget_bias(word_lm.lstm, 1.0)
STEPS_W = 200 if FAST_RUN else 1500
hist_w, _ = train_lm(word_lm, w_train, w_val, STEPS_W, 32, 3e-3, max(1, STEPS_W // 4), set(), None)
val_nats_per_word = hist_w["val"][-1]
print(f"validation perplexity per word: {math.exp(val_nats_per_word):.1f} "
      f"| equivalent per character (÷ mean word length incl. space): "
      f"{math.exp(val_nats_per_word / (np.mean([len(w) for w in words]) + 1)):.2f}")

prompt_w = word_ids[:5]
generated = sample_ids(word_lm, prompt_w, length=40, temperature=0.8)
print("\n" + " ".join(vocab_w[i] for i in prompt_w.tolist() + generated))

**Exercise 4.** Dropout between LSTM layers is the standard regulariser for recurrent language models. Take a *small* training slice (the first 8,000 characters) so that a two-layer model overfits within the budget, train it with `dropout=0.0` and `dropout=0.5`, and compare training and validation perplexities. What does the gap tell you, and why does `nn.LSTM` apply dropout only between layers and not inside the recurrence?

**Solution to Exercise 4**

On eight thousand characters a two-layer LSTM memorises quickly: the training perplexity keeps dropping while the validation perplexity stalls or rises. Dropout between the layers narrows the gap, at the cost of a slightly higher training loss. Applying ordinary dropout *inside* the recurrence, with a fresh mask at every time step, would compound the noise over the sequence and destroy the memory; the accepted way to regularise the recurrent weights is *variational* dropout, which uses the same mask at every step (Gal and Ghahramani, 2016), and it is not what `nn.LSTM`'s `dropout` argument does.

In [ ]:
small_train = make_streams(ids[:min(8_000, n_split)], 16).to(device)      # 16 x 500 chars: ~14 passes over the slice per 100 steps
small_val = make_streams(val_ids[:4000], 8).to(device)
STEPS_DO = 200 if FAST_RUN else 1500
for p_drop in [0.0, 0.5]:
    torch.manual_seed(SEED)
    m = CharLSTM(VOCAB_SIZE, emb_dim=32, hidden_size=128, num_layers=2, dropout=p_drop).to(device)
    set_forget_bias(m.lstm, 1.0)
    hist_do, _ = train_lm(m, small_train, small_val, STEPS_DO, 64, 3e-3, STEPS_DO, set(), None)
    train_ppl = math.exp(evaluate_lm(m, small_train, 64))
    val_ppl_do = math.exp(hist_do["val"][-1])
    print(f"dropout {p_drop:.1f}: train perplexity {train_ppl:6.2f} | val perplexity {val_ppl_do:6.2f} "
          f"| gap {val_ppl_do - train_ppl:+.2f}\n")

**Exercise 5 (challenge).** Implement Bahdanau (additive) attention for the reversal task of Section 9. At each decoder step $t$ with previous decoder state $s_{t-1}$ and encoder outputs $h_1, \dots, h_L$, compute scores $e_{tj} = v^{\top} \tanh\!\left(W_a [\, s_{t-1};\, h_j \,]\right)$, weights $\alpha_{tj} = \operatorname{softmax}_j(e_{tj})$, and the context $c_t = \sum_j \alpha_{tj} h_j$; feed $[\,\text{emb}(y_{t-1});\, c_t\,]$ to a `GRUCell`. Train it with the same budget as the plain model, plot accuracy versus length for both, and display the attention weights for one example.

**Solution to Exercise 5**

The decoder must now be unrolled step by step in Python, because the context at step $t$ depends on $s_{t-1}$. Everything else, including `train_seq2seq` and `accuracy_by_length`, is reused unchanged since the model keeps the same `forward(src, dec_in)` and `greedy_decode(src, max_len)` interface. The accuracy curve should stay near 100% for all training lengths and degrade far more gracefully beyond them, and the attention map for a reversed sequence should be an anti-diagonal: output step $t$ attends to input position $L + 1 - t$, counting both from one.

In [ ]:
class AttnDecoder(nn.Module):
    """GRU decoder with Bahdanau (additive) attention over the encoder outputs."""

    def __init__(self, vocab_size, emb_dim, hidden_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.attn_W = nn.Linear(2 * hidden_size, hidden_size)
        self.attn_v = nn.Linear(hidden_size, 1, bias=False)
        self.cell = nn.GRUCell(emb_dim + hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, vocab_size)

    def step(self, y_prev, s, enc_out):
        # y_prev: (B,), s: (B, H), enc_out: (B, L, H)
        s_rep = s[:, None, :].expand(-1, enc_out.shape[1], -1)                       # (B, L, H)
        scores = self.attn_v(torch.tanh(self.attn_W(torch.cat([s_rep, enc_out], dim=-1)))).squeeze(-1)  # (B, L)
        alpha = F.softmax(scores, dim=-1)
        context = torch.bmm(alpha[:, None, :], enc_out).squeeze(1)                    # (B, H)
        s = self.cell(torch.cat([self.emb(y_prev), context], dim=-1), s)
        return self.out(s), s, alpha

    def forward(self, dec_in, s, enc_out):                                            # teacher forcing
        logits, alphas = [], []
        for t in range(dec_in.shape[1]):
            lg, s, a = self.step(dec_in[:, t], s, enc_out)
            logits.append(lg); alphas.append(a)
        return torch.stack(logits, dim=1), torch.stack(alphas, dim=1)


class AttnSeq2Seq(nn.Module):
    def __init__(self, vocab_size, emb_dim=32, hidden_size=64):
        super().__init__()
        self.encoder = Encoder(vocab_size, emb_dim, hidden_size)
        self.decoder = AttnDecoder(vocab_size, emb_dim, hidden_size)

    def forward(self, src, dec_in):
        enc_out, h = self.encoder(src)
        logits, _ = self.decoder(dec_in, h[0], enc_out)
        return logits

    @torch.no_grad()
    def greedy_decode(self, src, max_len, return_attention=False):
        was_training = self.training
        self.eval()
        enc_out, h = self.encoder(src)
        s = h[0]
        y = torch.full((src.shape[0],), SOS, dtype=torch.long, device=src.device)
        outputs, alphas = [], []
        for _ in range(max_len):
            logits, s, a = self.decoder.step(y, s, enc_out)
            y = logits.argmax(dim=-1)
            outputs.append(y); alphas.append(a)
        self.train(was_training)
        out = torch.stack(outputs, dim=1)
        return (out, torch.stack(alphas, dim=1)) if return_attention else out


torch.manual_seed(SEED); random.seed(SEED)
attn_s2s = AttnSeq2Seq(V_S2S, hidden_size=64 if FAST_RUN else 128).to(device)
print(f"training attention seq2seq on lengths 3..{L_MAX_S2S}")
attn_losses = train_seq2seq(attn_s2s, STEPS_S2S, L_MAX_S2S)
acc_attn = accuracy_by_length(attn_s2s, EVAL_LENGTHS)
for L, a_plain, a_attn in zip(EVAL_LENGTHS, acc_s2s, acc_attn):
    print(f"length {L:2d}: plain {a_plain:.3f} | attention {a_attn:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(EVAL_LENGTHS, acc_s2s, marker="o", label="plain seq2seq")
axes[0].plot(EVAL_LENGTHS, acc_attn, marker="s", label="with Bahdanau attention")
axes[0].axvline(L_MAX_S2S + 0.5, color="gray", linestyle=":", label="max training length")
axes[0].set_xlabel("sequence length"); axes[0].set_ylabel("exact-match accuracy"); axes[0].set_ylim(0, 1.05)
axes[0].set_title("Reversal task"); axes[0].legend()

L_show = 8
src, _, _ = make_reverse_batch(1, L_show)
pred, alphas = attn_s2s.greedy_decode(src.to(device), L_show, return_attention=True)
im = axes[1].imshow(alphas[0].cpu().numpy(), cmap="viridis", aspect="auto")
axes[1].set_xticks(range(L_show)); axes[1].set_xticklabels(show_tokens(src[0]).split())
axes[1].set_yticks(range(L_show)); axes[1].set_yticklabels(show_tokens(pred[0]).split())
axes[1].set_xlabel("source position (input digit)"); axes[1].set_ylabel("output step (generated digit)")
axes[1].set_title("Attention weights"); axes[1].grid(False)
fig.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout(); plt.show()

## Summary

- Sequential data is stored as `(batch, time, features)` tensors; a recurrent network applies the same update $h_t = f(x_t, h_{t-1})$ at every step, which gives it a fixed parameter count for any length and makes it sensitive to order.
- The vanilla RNN, $h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b)$, is a deep network unrolled in time; our from-scratch implementation matches `nn.RNN` to floating-point precision.
- Backpropagation through time multiplies one Jacobian $\operatorname{diag}(1 - h_i^2) W_{hh}$ per step; the norm of the product is bounded by $(\sigma_{\max}(W_{hh}))^{\text{lag}}$, so gradients vanish exponentially when $\sigma_{\max} < 1$ and can explode otherwise. We measured this directly with `torch.autograd.grad` on intermediate states.
- Gradient clipping by norm bounds the update while preserving its direction and is essential for recurrent training; truncated BPTT bounds memory by detaching the carried state between chunks.
- The LSTM adds an additively updated cell state gated by learned sigmoids; along the cell path the gradient scales like $\prod f_t$, a learnable rate, which is why a forget-gate bias of 1–2 helps and why LSTMs and GRUs learn long-range dependencies where vanilla RNNs fail (Section 5).
- A character-level LSTM trained with truncated BPTT on Tiny Shakespeare learns spelling, layout and words from next-character prediction alone; perplexity measures its uncertainty and temperature controls the trade-off between conservative and diverse samples.
- Time-series forecasting becomes supervised learning through windowing; scalers must be fitted on the training period only, splits must respect time, the naive forecast is the baseline to beat, and multi-step rollouts accumulate error.
- Variable-length batches require padding *and* a mechanism to ignore it: `pack_padded_sequence` for recurrent layers, masks for pooling and losses.
- Encoder–decoder models with teacher forcing solve unaligned sequence-to-sequence tasks but compress the whole input into one vector; accuracy decays with length, and attention, which lets the decoder read all encoder states, removes the bottleneck and leads directly to Notebook 09 · Attention and Transformers.

## Further reading

- Hochreiter, S. and Schmidhuber, J., 1997. *Long Short-Term Memory*. Neural Computation 9(8).
- Bengio, Y., Simard, P. and Frasconi, P., 1994. *Learning Long-Term Dependencies with Gradient Descent is Difficult*. IEEE Transactions on Neural Networks 5(2).
- Pascanu, R., Mikolov, T. and Bengio, Y., 2013. *On the Difficulty of Training Recurrent Neural Networks*. ICML.
- Cho, K. et al., 2014. *Learning Phrase Representations using RNN Encoder–Decoder for Statistical Machine Translation*. EMNLP.
- Sutskever, I., Vinyals, O. and Le, Q. V., 2014. *Sequence to Sequence Learning with Neural Networks*. NeurIPS.
- Bahdanau, D., Cho, K. and Bengio, Y., 2015. *Neural Machine Translation by Jointly Learning to Align and Translate*. ICLR.
- Jozefowicz, R., Zaremba, W. and Sutskever, I., 2015. *An Empirical Exploration of Recurrent Network Architectures*. ICML.
- Gal, Y. and Ghahramani, Z., 2016. *A Theoretically Grounded Application of Dropout in Recurrent Neural Networks*. NeurIPS.
- Karpathy, A., 2015. *The Unreasonable Effectiveness of Recurrent Neural Networks* (blog essay); Olah, C., 2015. *Understanding LSTM Networks* (blog essay).
- Goodfellow, I., Bengio, Y. and Courville, A., 2016. *Deep Learning*, Chapter 10 (Sequence Modeling: Recurrent and Recursive Nets). MIT Press.
- Zhang, A., Lipton, Z. C., Li, M. and Smola, A. J., *Dive into Deep Learning*, Chapters 9–10 (Recurrent Neural Networks; Modern RNNs).
- PyTorch documentation: `torch.nn.LSTM`, `torch.nn.GRU`, `torch.nn.utils.rnn` — https://docs.pytorch.org/docs/stable/